# TimeGAN Data Augmentation - Standalone Pipeline
### Credit Rating Prediction - Kaggle Environment

**Objective:** Complete pipeline for data splitting, preprocessing, and TimeGAN-based sequence augmentation

**Key Principles:**
- Standalone: Always load source dataset and split inside this notebook
- TimeGAN learns temporal dynamics from train data only (anti-leakage protocol)
- Sequence-first augmentation: build windows by `ticker` and `rating_date`
- Sliding-window extraction with configurable `seq_len` and `sequence_stride`
- Minority-focused class balancing with per-class TimeGAN generation
- Flatten synthetic sequences back to tabular format for downstream compatibility

## Ä‘Å¸Ââ‚¬ TIMEGAN MODE (SEQUENCE-AWARE)

This notebook now uses **TimeGAN** instead of CTGAN to generate synthetic data that respects temporal structure.

### What Changed:
- **3D sequence modeling:** `(num_windows, seq_len, num_features)`
- **Temporal grouping:** sorted by `ticker` and `rating_date`
- **Class-focused generation:** train TimeGAN per eligible minority class
- **Tabular compatibility:** generated sequences are unrolled back into row-wise records

### Why It Matters:
- CTGAN generates independent rows (tabular-only)
- TimeGAN models transitions across time steps, better aligned with time-aware downstream models

### Expected Outputs:
- `splits/train_augmented_timegan.csv` - primary augmented train set
- `splits/train_augmented_ctgan.csv` - compatibility alias for existing downstream scripts
- `models/timegan/timegan_training_registry.json` - model/training metadata
- `reports/timegan_augmentation_report.md` - generation summary

## 1. Setup & Dependencies

In [ ]:
# Install dependencies for TimeGAN pipeline (Kaggle notebook mode)
%pip install -q --upgrade pip
%pip install -q "tsgm[tensorflow]" scikit-learn pandas numpy scipy matplotlib seaborn

print("Dependencies installed successfully")
print("If needed, restart the kernel and run all cells from the top")

In [5]:
import re
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import json
import warnings
from datetime import datetime
import torch

# Reduce TensorFlow/XLA startup logs in notebook output
os.environ.setdefault("KERAS_BACKEND", "tensorflow")
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "3")
os.environ.setdefault("ABSL_CPP_MIN_LOG_LEVEL", "3")

# Default to CPU to avoid CUDA plugin registration noise.
# Set TIMEGAN_ENABLE_GPU=1 before running notebook to enable GPU.
os.environ.setdefault("TIMEGAN_ENABLE_GPU", "0")
if os.environ.get("TIMEGAN_ENABLE_GPU", "0") != "1":
    os.environ["CUDA_VISIBLE_DEVICES"] = "-1"

# TimeGAN imports (tsgm)
TIMEGAN_AVAILABLE = True
TIMEGAN_IMPORT_ERROR = None
try:
    import tsgm
    from tsgm.models.timeGAN import TimeGAN
except Exception as e:
    tsgm = None
    TimeGAN = None
    TIMEGAN_AVAILABLE = False
    TIMEGAN_IMPORT_ERROR = str(e)

# Sklearn imports
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer
from scipy import stats
from scipy.spatial.distance import jensenshannon

warnings.filterwarnings('ignore')

# Set random seeds for reproducibility
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

try:
    import tensorflow as tf
    tf.random.set_seed(RANDOM_SEED)
    tf.get_logger().setLevel("ERROR")
except Exception:
    tf = None

# Display settings
pd.set_option('display.max_columns', None)
plt.style.use('seaborn-v0_8-darkgrid')

print("Libraries imported successfully")
print(f"Random seed: {RANDOM_SEED}")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available (PyTorch): {torch.cuda.is_available()}")
print(f"TensorFlow available: {tf is not None}")
print(f"TimeGAN import available: {TIMEGAN_AVAILABLE}")
print(f"TIMEGAN_ENABLE_GPU={os.environ.get('TIMEGAN_ENABLE_GPU', '0')}")
if TIMEGAN_AVAILABLE and tsgm is not None:
    print(f"TSGM version: {getattr(tsgm, '__version__', 'unknown')}")
if not TIMEGAN_AVAILABLE:
    print(f"TimeGAN import error: {TIMEGAN_IMPORT_ERROR}")

Libraries imported successfully
Random seed: 42
PyTorch version: 2.10.0+cpu
CUDA available (PyTorch): False
TensorFlow available: False
TimeGAN import available: False
TIMEGAN_ENABLE_GPU=0
TimeGAN import error: No module named 'tsgm'


## 2. Configuration & Paths

In [ ]:
# Configure paths for local workspace and Kaggle
from pathlib import Path
from datetime import datetime

PROJECT_ROOT_CANDIDATES = [Path.cwd(), Path.cwd().parent]
PROJECT_ROOT = next((p for p in PROJECT_ROOT_CANDIDATES if (p / 'data').exists()), Path.cwd())
DEFAULT_DATA_FILE = 'merged_credit_rating_common_3groups.csv'
LOCAL_INPUT_PATH = PROJECT_ROOT / 'data' / 'processed'
KAGGLE_INPUT_PATH = Path('/kaggle/input/datasets/tailength/corporate-credit-rating')

if (LOCAL_INPUT_PATH / DEFAULT_DATA_FILE).exists():
    RUN_MODE = 'local'
    INPUT_PATH = LOCAL_INPUT_PATH
elif (KAGGLE_INPUT_PATH / DEFAULT_DATA_FILE).exists():
    RUN_MODE = 'kaggle'
    INPUT_PATH = KAGGLE_INPUT_PATH
else:
    RUN_MODE = 'local_missing_data'
    INPUT_PATH = LOCAL_INPUT_PATH

OUTPUT_PATH = Path('/kaggle/working') if RUN_MODE == 'kaggle' else (PROJECT_ROOT / 'data' / 'external' / 'timegan_3groups_output')

SPLITS_DIR = OUTPUT_PATH / 'splits'
MODELS_DIR = OUTPUT_PATH / 'models' / 'timegan'
REPORTS_DIR = OUTPUT_PATH / 'reports'
for d in [SPLITS_DIR, MODELS_DIR, REPORTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# ---- CONFIGURATION v5 (strict downstream-window eligibility) ----
# Changes vs v4:
# 1. Enforce minimum records per synthetic ticker for downstream windows
# 2. Optionally force all synthetic tickers to satisfy downstream window requirement
# 3. Keep anti-leakage and sequence-aware generation constraints
CONFIG = {
    'random_seed': RANDOM_SEED,
    'train_ratio': 0.8, 'val_ratio': 0.1, 'test_ratio': 0.1, 'stratify': True,
    'data_file': 'merged_credit_rating_common_3groups.csv',
    'target_column': 'rating_detail',
    'target_is_encoded_label': True, 'target_min_label': 0, 'target_max_label': 2,
    'date_column': 'rating_date', 'entity_column': 'ticker',

    # Split strategy (new default)
    'split_method': 'walk_forward_expanding',
    'walk_forward_min_train_years': 5,
    'walk_forward_val_years': 1,
    'walk_forward_test_years': 1,
    'walk_forward_min_train_samples': 300,
    'walk_forward_active_fold': 'latest',

    # Sequence
    'seq_len': 8,
    'sequence_stride': 1,
    'min_history_for_company': 4,
    'allow_padded_windows': False,
    'timegan_use_gpu': True,

    # Downstream compatibility audit (Transformer/LSTM sliding-window)
    # INPUT_SIZE=8 + HORIZON=1 => minimum 9 rows per ticker
    'downstream_min_history_for_window': 9,
    'synthetic_min_points_per_ticker': 9,
    'downstream_audit_min_synth_window_ratio': 0.10,
    'downstream_enforce_all_synthetic_tickers_eligible': True,
    # Preferred block length when materializing synthetic sequences
    'mixup_sequence_block_len': 9,

    # Synthetic date/ticker constraints
    'date_generation_mode': 'empirical_cadence',
    'max_date_step_days': 365,

    # Budget optimization
    'budget_optimization_enabled': False,
    'budget_optimization_mode': 'val_proxy',
    'budget_ratio_candidates': [0.25, 0.35, 0.45],
    'budget_objective': {
        'macro_f1_weight': 0.7,
        'balanced_accuracy_weight': 0.3
    },

    # Minority support v2
    'priority_class_labels': [0, 1],
    'priority_boost_multiplier': 3.0,
    'priority_min_count': 500,
    'priority_max_oversample_factor': 180.0,
    'max_synthetic_ratio': 0.55,
    # Distressed-focused controls
    'distressed_label': 0,
    'distressed_target_ratio_vs_majority': 0.85,
    'distressed_min_synthetic': 700,
    'distressed_js_relax_multiplier': 1.6,
    'distressed_rf_relax_multiplier': 0.75,
    'distressed_windows_cap': 384,
    'timegan_batch_fallback': [24, 16, 8],
    'timegan_retry_window_caps': [900, 600, 400],
    'rf_n_jobs': 2,
    'rf_quality_estimators': 120,
    'rf_proxy_estimators': 120,

    # Balance strategy: 'log_ratio' | 'sqrt_flat' | 'median_flat'
    'balance_strategy': 'median_flat',

    # TimeGAN hyper-params
    'timegan_train_steps': 140, 'timegan_batch_size': 24,
    'timegan_learning_rate': 5e-4, 'timegan_noise_dim': 32,
    'timegan_layers_dim': 128, 'timegan_latent_dim': 24,
    'timegan_gamma': 1, 'timegan_module': 'gru', 'timegan_n_layers': 3,

    # Training stability
    'timegan_stability_multi_seed': False,
    'timegan_candidate_seeds': [42],
    'timegan_eval_windows': 16,
    'timegan_max_windows_per_model': 900,
    'timegan_generation_max_windows_per_class': 256,

    # Min windows guard
    'min_windows_per_class_for_timegan': 6,

    # Sector-conditional fallback
    'sector_column': 'sector', 'enable_sector_conditional': False,

    # Quality filters
    'quality_quantile_low': 0.005, 'quality_quantile_high': 0.995,
    'js_divergence_threshold': 0.20,
    'rf_quality_threshold': 0.50,

    # Ordinal MixUp fallback (sequence materialization happens later)
    'enable_ordinal_mixup_fallback': True,
    'mixup_alpha': 0.3, 'mixup_adjacent_only': True,

    # Hybrid rebalancing
    'enable_hybrid_rebalancing': True,
    'majority_cap_percentile': 80,
    'minority_floor': 150,
    'timestamp': datetime.now().isoformat()
}

timegan_gpu_runtime_ok = False
timegan_gpu_runtime_note = None
if CONFIG['timegan_use_gpu'] and tf is not None:
    try:
        gpus = tf.config.list_physical_devices('GPU')
        if len(gpus) > 0:
            for g in gpus:
                try:
                    tf.config.experimental.set_memory_growth(g, True)
                except Exception:
                    pass
            _ = tf.matmul(tf.random.uniform((64, 64)), tf.random.uniform((64, 64)))
            timegan_gpu_runtime_ok = True
            timegan_gpu_runtime_note = f"GPU OK (memory growth): {[g.name for g in gpus]}"
        else:
            CONFIG['timegan_use_gpu'] = False
            timegan_gpu_runtime_note = "No GPU found, CPU fallback"
    except Exception as e:
        CONFIG['timegan_use_gpu'] = False
        timegan_gpu_runtime_note = f"GPU check failed: {e}"
else:
    CONFIG['timegan_use_gpu'] = False
    timegan_gpu_runtime_note = "CPU mode" if tf is not None else "TF not available"

print("Configuration v5 loaded.")
print(f"Run mode: {RUN_MODE}")
print(f"Input path: {INPUT_PATH}")
print(f"Output path: {OUTPUT_PATH}")
print(json.dumps({k: v for k, v in CONFIG.items() if k != 'timestamp'}, indent=2))
print(timegan_gpu_runtime_note)
if not TIMEGAN_AVAILABLE:
    print("WARNING: TimeGAN not importable.")
with open(OUTPUT_PATH / 'config_timegan.json', 'w', encoding='utf-8') as f:
    json.dump(CONFIG, f, indent=2)


Configuration v5 loaded.
Run mode: local
Input path: e:\thesis\data\processed
Output path: e:\thesis\data\external\timegan_3groups_output
{
  "random_seed": 42,
  "train_ratio": 0.8,
  "val_ratio": 0.1,
  "test_ratio": 0.1,
  "stratify": true,
  "data_file": "merged_credit_rating_common_3groups.csv",
  "target_column": "rating_detail",
  "target_is_encoded_label": true,
  "target_min_label": 0,
  "target_max_label": 2,
  "date_column": "rating_date",
  "entity_column": "ticker",
  "split_method": "walk_forward_expanding",
  "walk_forward_min_train_years": 5,
  "walk_forward_val_years": 1,
  "walk_forward_test_years": 1,
  "walk_forward_min_train_samples": 300,
  "walk_forward_active_fold": "latest",
  "seq_len": 8,
  "sequence_stride": 1,
  "min_history_for_company": 4,
  "allow_padded_windows": false,
  "timegan_use_gpu": false,
  "downstream_min_history_for_window": 9,
  "synthetic_min_points_per_ticker": 9,
  "downstream_audit_min_synth_window_ratio": 0.1,
  "downstream_enforce_all_

## 3. Load Data & Split (Walk-Forward Expanding Window)

**Strategy:**

- Always load source dataset from INPUT_PATH
- Parse and clean target/date columns
- Build temporal folds with walk-forward expanding window
- Enforce strict anti-leakage by global date boundaries
- Keep latest fold as active train/val/test for downstream pipeline
- Save split artifacts for reproducibility and downstream comparison

In [7]:
print("=" * 70)
print("LOADING SOURCE DATA AND PERFORMING TEMPORAL SPLIT")
print("=" * 70)

# Load dataset from source path
data_file = INPUT_PATH / CONFIG['data_file']
if not data_file.exists():
    raise FileNotFoundError(
        f"Missing dataset: {data_file}. "
        f"Expected file name: {CONFIG['data_file']}. "
        "Check INPUT_PATH and dataset location."
    )
df = pd.read_csv(data_file)

print(f"\n Dataset loaded: {data_file}")
print(f"  Shape: {df.shape}")
print(f"  Columns: {len(df.columns)}")

if CONFIG['target_column'] not in df.columns:
    raise ValueError(f"Target column '{CONFIG['target_column']}' not found in dataset")

if CONFIG['date_column'] not in df.columns:
    raise ValueError(
        f"Date column '{CONFIG['date_column']}' not found. "
        "Walk-forward validation requires a valid temporal column."
    )

# Parse date column
df[CONFIG['date_column']] = pd.to_datetime(df[CONFIG['date_column']], errors='coerce')
missing_dates = int(df[CONFIG['date_column']].isna().sum())
if missing_dates > 0:
    print(f"  â  Found {missing_dates} rows with invalid {CONFIG['date_column']} and will drop them")
df = df.dropna(subset=[CONFIG['date_column']]).copy()
print(f"   Parsed {CONFIG['date_column']} to datetime")

def normalize_rating_text(value):
    """Normalize rating string for robust mapping."""
    if pd.isna(value):
        return np.nan
    return str(value).strip().upper().replace(' ', '')

# Normalize target column for encoded labels (expected range 0-2)
if CONFIG.get('target_is_encoded_label', False):
    target_col = CONFIG['target_column']
    target_numeric = pd.to_numeric(df[target_col], errors='coerce')
    non_na_numeric = int(target_numeric.notna().sum())

    if non_na_numeric == 0:
        print("  â  Target appears to be rating text. Applying fallback mapping to labels 0-2...")

        # 3-group mapping for merged_credit_rating_common_3groups.csv (worst -> best).
        # Labels are contiguous in [0, 2].
        ordered_ratings = [
            'DISTRESSED', 'HY', 'IG'
        ]
        rating_to_label = {rating: idx for idx, rating in enumerate(ordered_ratings)}

        normalized_target = df[target_col].apply(normalize_rating_text)
        mapped_target = normalized_target.map(rating_to_label)

        unknown_mask = mapped_target.isna() & normalized_target.notna()
        unknown_ratings = sorted(normalized_target[unknown_mask].unique().tolist())
        if unknown_ratings:
            sample_unknown = unknown_ratings[:10]
            raise ValueError(
                "Found unmapped rating labels in target: "
                f"{sample_unknown}. Please extend rating_to_label mapping in Cell 9."
            )

        df[target_col] = mapped_target
        print(f"   Fallback mapping completed with {int(df[target_col].notna().sum())} labeled rows")
    else:
        before_na = int(df[target_col].isna().sum())
        df[target_col] = target_numeric
        after_na = int(df[target_col].isna().sum())
        if after_na > before_na:
            print(f"  â  Coercion added {after_na - before_na} NaN values in target")

    min_label = int(CONFIG.get('target_min_label', 0))
    max_label = int(CONFIG.get('target_max_label', 2))
    in_range_mask = df[target_col].between(min_label, max_label) | df[target_col].isna()
    out_of_range = int((~in_range_mask).sum())
    if out_of_range > 0:
        print(f"  â  Found {out_of_range} rows with target outside [{min_label}, {max_label}] and will drop them")
    df = df[in_range_mask].copy()

# Remove rows with missing target
initial_rows = len(df)
df = df.dropna(subset=[CONFIG['target_column']]).copy()
print(f"   Removed {initial_rows - len(df)} rows with missing target")

if CONFIG.get('target_is_encoded_label', False):
    df[CONFIG['target_column']] = df[CONFIG['target_column']].astype(int)
    uniq = sorted(df[CONFIG['target_column']].unique().tolist())
    print(f"   Target converted to int labels, unique classes: {len(uniq)}")
    if len(uniq) == 0:
        raise ValueError(
            "Target became empty after normalization/filtering. "
            "Check target mapping and source data in Cell 9."
        )
    print(f"   Label range in data: min={min(uniq)}, max={max(uniq)}")

# Sort by time once for deterministic temporal slicing
df = df.sort_values(CONFIG['date_column']).reset_index(drop=True)
print(f"   Final shape after cleaning: {df.shape}")

def point_in_time_sanity_check(frame, rating_date_col):
    """Basic point-in-time check for date-like feature columns."""
    date_like_cols = [
        c for c in frame.columns
        if c != rating_date_col and ('date' in c.lower() or 'time' in c.lower())
    ]
    if len(date_like_cols) == 0:
        print("  â  Point-in-time check: no additional date-like feature columns found.")
        print("    Please manually verify financial ratios are known before rating_date.")
        return {}

    violations = {}
    for col in date_like_cols:
        probe = pd.to_datetime(frame[col], errors='coerce')
        comparable = probe.notna() & frame[rating_date_col].notna()
        if int(comparable.sum()) == 0:
            continue
        n_future = int((probe[comparable] > frame.loc[comparable, rating_date_col]).sum())
        if n_future > 0:
            violations[col] = n_future

    if len(violations) == 0:
        print("   Point-in-time check: no obvious future-date violations detected.")
    else:
        print("  â  Point-in-time check found potential future leakage:")
        for k, v in violations.items():
            print(f"    - {k}: {v} rows where {k} > {rating_date_col}")
    return violations

def build_walk_forward_expanding_folds(
    frame, date_col, entity_col, min_train_years, val_years, test_years, min_train_samples
 ):
    """Build expanding-window folds by calendar year using global temporal cutoffs."""
    working = frame.copy()
    working['_wf_year'] = working[date_col].dt.year.astype(int)
    years = sorted(working['_wf_year'].unique().tolist())

    folds = []
    fold_id = 0
    max_train_end_idx = len(years) - (val_years + test_years) - 1
    for train_end_idx in range(min_train_years - 1, max_train_end_idx + 1):
        train_years = years[:train_end_idx + 1]
        val_year_block = years[train_end_idx + 1: train_end_idx + 1 + val_years]
        test_year_block = years[train_end_idx + 1 + val_years: train_end_idx + 1 + val_years + test_years]

        if len(val_year_block) < val_years or len(test_year_block) < test_years:
            continue

        train_df = working[working['_wf_year'].isin(train_years)].drop(columns=['_wf_year']).copy()
        val_df = working[working['_wf_year'].isin(val_year_block)].drop(columns=['_wf_year']).copy()
        test_df = working[working['_wf_year'].isin(test_year_block)].drop(columns=['_wf_year']).copy()

        if len(train_df) < min_train_samples or len(val_df) == 0 or len(test_df) == 0:
            continue

        # Strict temporal anti-leakage checks
        train_max = train_df[date_col].max()
        val_min = val_df[date_col].min()
        val_max = val_df[date_col].max()
        test_min = test_df[date_col].min()
        if not (train_max < val_min and val_max < test_min):
            continue

        fold_id += 1
        train_tickers = set(train_df[entity_col].dropna().astype(str).unique()) if entity_col in train_df.columns else set()
        test_tickers = set(test_df[entity_col].dropna().astype(str).unique()) if entity_col in test_df.columns else set()
        new_test_tickers = sorted(list(test_tickers - train_tickers))

        folds.append({
            'fold_id': fold_id,
            'train': train_df,
            'val': val_df,
            'test': test_df,
            'meta': {
                'train_years': train_years,
                'val_years': val_year_block,
                'test_years': test_year_block,
                'train_end_date': str(train_max.date()),
                'val_start_date': str(val_min.date()),
                'val_end_date': str(val_max.date()),
                'test_start_date': str(test_min.date()),
                'train_size': int(len(train_df)),
                'val_size': int(len(val_df)),
                'test_size': int(len(test_df)),
                'new_test_ticker_count': int(len(new_test_tickers))
            }
        })

    return folds, years

# Execute split
split_method = str(CONFIG.get('split_method', 'walk_forward_expanding')).lower()
if split_method == 'walk_forward_expanding':
    print("\n=== Walk-Forward Validation (Expanding Window) ===")
    pt_violations = point_in_time_sanity_check(df, CONFIG['date_column'])

    wf_folds, wf_years = build_walk_forward_expanding_folds(
        frame=df,
        date_col=CONFIG['date_column'],
        entity_col=CONFIG['entity_column'],
        min_train_years=int(CONFIG.get('walk_forward_min_train_years', 5)),
        val_years=int(CONFIG.get('walk_forward_val_years', 1)),
        test_years=int(CONFIG.get('walk_forward_test_years', 1)),
        min_train_samples=int(CONFIG.get('walk_forward_min_train_samples', 300))
    )

    if len(wf_folds) == 0:
        raise ValueError(
            "No valid walk-forward folds were created. "
            "Adjust walk_forward_* config or inspect date coverage."
        )

    active_spec = CONFIG.get('walk_forward_active_fold', 'latest')
    if isinstance(active_spec, str) and active_spec.lower() == 'latest':
        active_fold = wf_folds[-1]
    else:
        active_idx = int(active_spec)
        if active_idx < 1 or active_idx > len(wf_folds):
            raise ValueError(
                f"walk_forward_active_fold={active_idx} is out of range 1..{len(wf_folds)}"
            )
        active_fold = wf_folds[active_idx - 1]

    train = active_fold['train'].copy()
    val = active_fold['val'].copy()
    test = active_fold['test'].copy()

    print(f"Total years available: {wf_years}")
    print(f"Created folds: {len(wf_folds)}")
    for fold in wf_folds:
        m = fold['meta']
        print(
            f"  Fold {fold['fold_id']}: "
            f"train={m['train_years']} | val={m['val_years']} | test={m['test_years']} "
            f"sizes=({m['train_size']}, {m['val_size']}, {m['test_size']}) "
            f"new_test_tickers={m['new_test_ticker_count']}"
        )

    am = active_fold['meta']
    print("\n Active fold selected:")
    print(f"  Fold ID: {active_fold['fold_id']}")
    print(f"  Train years: {am['train_years']}")
    print(f"  Val years:   {am['val_years']}")
    print(f"  Test years:  {am['test_years']}")
    print(f"  Train size: {len(train)}")
    print(f"  Val size:   {len(val)}")
    print(f"  Test size:  {len(test)}")

    # Cold-start indicator: tickers in test never seen in train
    if CONFIG['entity_column'] in train.columns and CONFIG['entity_column'] in test.columns:
        train_tickers = set(train[CONFIG['entity_column']].dropna().astype(str).unique())
        test_tickers = set(test[CONFIG['entity_column']].dropna().astype(str).unique())
        cold_start_tickers = sorted(list(test_tickers - train_tickers))
        print(f"  Cold-start tickers in active test: {len(cold_start_tickers)}")

    TRAIN_VAL_TEST_SPLIT_INFO = {
        'split_method': 'walk_forward_expanding',
        'active_fold_id': int(active_fold['fold_id']),
        'n_folds': int(len(wf_folds)),
        'train_years': am['train_years'],
        'val_years': am['val_years'],
        'test_years': am['test_years'],
        'train_end_date': am['train_end_date'],
        'val_start_date': am['val_start_date'],
        'test_start_date': am['test_start_date'],
        'point_in_time_violations': pt_violations
    }

    fold_summary = pd.DataFrame([
        {
            'fold_id': f['fold_id'],
            'train_years': ','.join([str(y) for y in f['meta']['train_years']]),
            'val_years': ','.join([str(y) for y in f['meta']['val_years']]),
            'test_years': ','.join([str(y) for y in f['meta']['test_years']]),
            'train_size': f['meta']['train_size'],
            'val_size': f['meta']['val_size'],
            'test_size': f['meta']['test_size'],
            'new_test_ticker_count': f['meta']['new_test_ticker_count']
        }
        for f in wf_folds
    ])
    fold_summary.to_csv(REPORTS_DIR / 'walk_forward_folds_summary.csv', index=False)
    print(f" Walk-forward fold summary saved: {REPORTS_DIR / 'walk_forward_folds_summary.csv'}")
else:
    print("\n=== Random Split Fallback ===")
    stratify_series = df[CONFIG['target_column']] if CONFIG.get('stratify', True) else None
    min_class_count = int(stratify_series.value_counts().min()) if stratify_series is not None else None
    if stratify_series is not None and min_class_count < 2:
        print("  â  Some classes have <2 samples, disabling stratified split")
        stratify_series = None

    train_val, test = train_test_split(
        df,
        test_size=CONFIG['test_ratio'],
        random_state=CONFIG['random_seed'],
        stratify=stratify_series
    )

    val_ratio_adjusted = CONFIG['val_ratio'] / (1 - CONFIG['test_ratio'])
    stratify_train_val = train_val[CONFIG['target_column']] if stratify_series is not None else None
    if stratify_train_val is not None and int(stratify_train_val.value_counts().min()) < 2:
        stratify_train_val = None

    train, val = train_test_split(
        train_val,
        test_size=val_ratio_adjusted,
        random_state=CONFIG['random_seed'],
        stratify=stratify_train_val
    )

    TRAIN_VAL_TEST_SPLIT_INFO = {
        'split_method': 'random',
        'train_ratio': CONFIG['train_ratio'],
        'val_ratio': CONFIG['val_ratio'],
        'test_ratio': CONFIG['test_ratio']
    }

# Check class distribution
print(f"\n=== Class Distribution ===")
for name, split_df in [('Train', train), ('Val', val), ('Test', test)]:
    print(f"\n{name}:")
    print(split_df[CONFIG['target_column']].value_counts().sort_index())

# Save raw splits for reproducibility before preprocessing
train.to_csv(SPLITS_DIR / 'train_raw.csv', index=False)
val.to_csv(SPLITS_DIR / 'val_raw.csv', index=False)
test.to_csv(SPLITS_DIR / 'test_raw.csv', index=False)
print(f"\n Raw splits saved to {SPLITS_DIR}")

with open(REPORTS_DIR / 'split_metadata.json', 'w', encoding='utf-8') as f:
    json.dump(TRAIN_VAL_TEST_SPLIT_INFO, f, indent=2, ensure_ascii=False, default=str)
print(f" Split metadata saved: {REPORTS_DIR / 'split_metadata.json'}")

preprocessed = False

print(f"\n{'='*70}")
print(f"Data loading complete. Preprocessed: {preprocessed}")
print(f"{'='*70}")

LOADING SOURCE DATA AND PERFORMING TEMPORAL SPLIT

 Dataset loaded: e:\thesis\data\processed\merged_credit_rating_common_3groups.csv
  Shape: (8680, 19)
  Columns: 19
   Parsed rating_date to datetime
  â  Target appears to be rating text. Applying fallback mapping to labels 0-2...
   Fallback mapping completed with 8680 labeled rows
   Removed 0 rows with missing target
   Target converted to int labels, unique classes: 3
   Label range in data: min=0, max=2
   Final shape after cleaning: (8680, 19)

=== Walk-Forward Validation (Expanding Window) ===
  â  Point-in-time check: no additional date-like feature columns found.
    Please manually verify financial ratios are known before rating_date.
Total years available: [2005, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016]
Created folds: 3
  Fold 1: train=[2005, 2009, 2010, 2011, 2012] | val=[2013] | test=[2014] sizes=(1645, 1509, 2096) new_test_tickers=138
  Fold 2: train=[2005, 2009, 2010, 2011, 2012, 2013] | val=[2014] | test=[2015

In [ ]:
# 4. Preprocessing Pipeline (fit on train only)

print("\n" + "=" * 70)

print("PREPROCESSING PIPELINE")

print("Anti-leakage: all transforms fit only on train set")

print("=" * 70)


## 4.1 Execute Preprocessing



Run the next code cell to fit preprocessing on train and transform val/test.


In [ ]:
if not preprocessed:

    print("\n" + "=" * 70)

    print("PERFORMING PREPROCESSING PIPELINE")

    print("=" * 70)



    # Identify column types

    NUMERIC_COLS = train.select_dtypes(include=[np.number]).columns.tolist()

    CATEGORICAL_COLS = train.select_dtypes(include=['object', 'category']).columns.tolist()



    # Remove identifier and target columns

    IDENTIFIER_COLS = []

    if 'company_name' in train.columns:

        IDENTIFIER_COLS.append('company_name')

    if 'ticker' in train.columns:

        IDENTIFIER_COLS.append('ticker')

    if CONFIG['date_column'] in train.columns:

        IDENTIFIER_COLS.append(CONFIG['date_column'])



    FEATURE_NUMERIC = [c for c in NUMERIC_COLS if c not in [CONFIG['target_column']] + IDENTIFIER_COLS]

    FEATURE_CATEGORICAL = [c for c in CATEGORICAL_COLS if c not in [CONFIG['target_column']] + IDENTIFIER_COLS]



    print(f"\n Feature identification:")

    print(f"  Numeric features: {len(FEATURE_NUMERIC)}")

    print(f"  Categorical features: {len(FEATURE_CATEGORICAL)}")



    PREPROCESSING_META = {

        'numeric_features': FEATURE_NUMERIC,

        'categorical_features': FEATURE_CATEGORICAL,

        'transformations': {}

    }



    # 1. Missing value imputation

    print(f"\n=== Missing Value Imputation ===")

    numeric_imputer = SimpleImputer(strategy='median')

    train[FEATURE_NUMERIC] = numeric_imputer.fit_transform(train[FEATURE_NUMERIC])

    val[FEATURE_NUMERIC] = numeric_imputer.transform(val[FEATURE_NUMERIC])

    test[FEATURE_NUMERIC] = numeric_imputer.transform(test[FEATURE_NUMERIC])

    print(" Numeric imputation (median)")



    if FEATURE_CATEGORICAL:

        categorical_imputer = SimpleImputer(strategy='constant', fill_value='__MISSING__')

        train[FEATURE_CATEGORICAL] = categorical_imputer.fit_transform(train[FEATURE_CATEGORICAL])

        val[FEATURE_CATEGORICAL] = categorical_imputer.transform(val[FEATURE_CATEGORICAL])

        test[FEATURE_CATEGORICAL] = categorical_imputer.transform(test[FEATURE_CATEGORICAL])

        print(" Categorical imputation (__MISSING__)")



    # 2. Log transformation

    print(f"\n=== Log Transformation for Skewness Reduction ===")

    SKEWNESS_THRESHOLD = 1.0

    LOG_TRANSFORM_META = {}



    for col in FEATURE_NUMERIC:

        pre_skew = stats.skew(train[col].dropna())



        if abs(pre_skew) < SKEWNESS_THRESHOLD:

            transform_type = 'none'

        elif (train[col] >= 0).all():

            transform_type = 'log1p'

            train[col] = np.log1p(train[col])

            val[col] = np.log1p(val[col])

            test[col] = np.log1p(test[col])

        else:

            transform_type = 'signed_log1p'

            train[col] = np.sign(train[col]) * np.log1p(np.abs(train[col]))

            val[col] = np.sign(val[col]) * np.log1p(np.abs(val[col]))

            test[col] = np.sign(test[col]) * np.log1p(np.abs(test[col]))



        post_skew = stats.skew(train[col].dropna())

        LOG_TRANSFORM_META[col] = {

            'transform_type': transform_type,

            'pre_skewness': float(pre_skew),

            'post_skewness': float(post_skew)

        }



    transformed_cols = [k for k, v in LOG_TRANSFORM_META.items() if v['transform_type'] != 'none']

    print(f" Log transformation applied ({len(transformed_cols)} columns)")

    PREPROCESSING_META['transformations']['log_transform'] = LOG_TRANSFORM_META



    # 3. Scaling

    print(f"\n=== Scaling ===")

    scaler = StandardScaler()

    train[FEATURE_NUMERIC] = scaler.fit_transform(train[FEATURE_NUMERIC])

    val[FEATURE_NUMERIC] = scaler.transform(val[FEATURE_NUMERIC])

    test[FEATURE_NUMERIC] = scaler.transform(test[FEATURE_NUMERIC])

    print(" StandardScaler applied")



    # 4. Categorical encoding

    if FEATURE_CATEGORICAL:

        print(f"\n=== Categorical Encoding ===")

        CATEGORICAL_MAPPINGS = {}



        for col in FEATURE_CATEGORICAL:

            le = LabelEncoder()

            train[col] = train[col].astype(str)

            val[col] = val[col].astype(str)

            test[col] = test[col].astype(str)



            # Fit on train plus fallback token to avoid unseen-category crashes

            train_with_missing = pd.concat([train[col], pd.Series(['__MISSING__'])], ignore_index=True)

            le.fit(train_with_missing)



            train[col] = le.transform(train[col])

            val[col] = val[col].apply(lambda x: x if x in le.classes_ else '__MISSING__')

            test[col] = test[col].apply(lambda x: x if x in le.classes_ else '__MISSING__')

            val[col] = le.transform(val[col])

            test[col] = le.transform(test[col])



            CATEGORICAL_MAPPINGS[col] = dict(zip(le.classes_.tolist(), le.transform(le.classes_).tolist()))



        PREPROCESSING_META['transformations']['categorical_encoding'] = CATEGORICAL_MAPPINGS

        print(f" Label encoding applied ({len(FEATURE_CATEGORICAL)} columns)")



    # Save preprocessing metadata

    with open(MODELS_DIR / 'preprocessing_metadata.json', 'w', encoding='utf-8') as f:

        json.dump(PREPROCESSING_META, f, indent=2, default=str)

    print(f"\n Preprocessing metadata saved")



    # Save preprocessed splits

    train.to_csv(SPLITS_DIR / 'train.csv', index=False)

    val.to_csv(SPLITS_DIR / 'val.csv', index=False)

    test.to_csv(SPLITS_DIR / 'test.csv', index=False)

    print(f" Preprocessed splits saved to {SPLITS_DIR}")



    print(f"\n{'='*70}")

    print("PREPROCESSING COMPLETE")

    print(f"{'='*70}")

else:

    print("\n Data already preprocessed, skipping preprocessing steps")
    if 'PREPROCESSING_META' not in globals():
        meta_path = MODELS_DIR / 'preprocessing_metadata.json'
        if not meta_path.exists():
            raise FileNotFoundError(
                f"Missing preprocessing metadata at {meta_path}. "
                "Run preprocessing first or set preprocessed=False."
            )
        with open(meta_path, 'r', encoding='utf-8') as f:
            PREPROCESSING_META = json.load(f)
        print(f" Loaded preprocessing metadata from {meta_path}")


# Identify features for later use

FEATURE_NUMERIC = PREPROCESSING_META['numeric_features']

FEATURE_CATEGORICAL = PREPROCESSING_META['categorical_features']


In [ ]:
# Analyze class distribution
print("\n=== Class Distribution ===")
full_class_index = list(range(int(CONFIG['target_min_label']), int(CONFIG['target_max_label']) + 1))

train_class_dist = (
    train[CONFIG['target_column']]
    .value_counts()
    .sort_index()
    .reindex(full_class_index, fill_value=0)
    .astype(int)
)
train_class_pct = (train_class_dist / max(train_class_dist.sum(), 1) * 100).round(2)

print("\nTrain set (count):")
print(train_class_dist)
nonzero_classes = train_class_dist[train_class_dist > 0]
if len(nonzero_classes) > 0:
    print(f"\nClass balance ratio (non-zero classes): {nonzero_classes.min() / nonzero_classes.max():.3f}")
else:
    print("\nClass balance ratio: N/A (all classes are zero)")

missing_classes = train_class_dist[train_class_dist == 0].index.tolist()
if missing_classes:
    print(f"Missing classes in train (shown as zero on chart): {missing_classes}")

# Subplot visualization: count + percentage
fig, axes = plt.subplots(1, 2, figsize=(18, 6), constrained_layout=True)

# Left: absolute counts
train_class_dist.plot(kind='bar', ax=axes[0], color='steelblue', edgecolor='black', linewidth=0.5)
axes[0].set_title('Class Distribution - Count (Train)')
axes[0].set_xlabel('Class')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=0)
axes[0].grid(axis='y', alpha=0.25)

# Right: percentage
train_class_pct.plot(kind='bar', ax=axes[1], color='coral', edgecolor='black', linewidth=0.5)
axes[1].set_title('Class Distribution - Percentage (Train)')
axes[1].set_xlabel('Class')
axes[1].set_ylabel('Percentage (%)')
axes[1].tick_params(axis='x', rotation=0)
axes[1].grid(axis='y', alpha=0.25)

# Save + show
plt.savefig(REPORTS_DIR / 'class_distribution_before_timegan_subplot.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Prepare Data for TimeGAN

TimeGAN requires sequence windows grouped by entity and ordered in time.

In [ ]:
# Prepare training data for TimeGAN using entity-based sequence windows
if not TIMEGAN_AVAILABLE:
    raise ImportError(f"TimeGAN import failed: {TIMEGAN_IMPORT_ERROR}")

target_col = CONFIG['target_column']
date_col = CONFIG['date_column']
entity_col = CONFIG['entity_column']

train_for_timegan = train.copy()

# Ensure required entity/date columns exist for sequence extraction
if entity_col not in train_for_timegan.columns:
    fallback_entity_col = 'company_name' if 'company_name' in train_for_timegan.columns else None
    if fallback_entity_col is not None:
        train_for_timegan[entity_col] = train_for_timegan[fallback_entity_col].astype(str)
        print(f"Missing '{entity_col}', fallback to '{fallback_entity_col}'")
    else:
        train_for_timegan[entity_col] = [f"ENTITY_{i:07d}" for i in range(len(train_for_timegan))]
        print(f"Missing '{entity_col}' and 'company_name', generated synthetic entity IDs")

if date_col not in train_for_timegan.columns:
    train_for_timegan[date_col] = pd.Timestamp('2000-01-01')
    print(f"Missing '{date_col}', fallback to constant timestamp")

train_for_timegan[date_col] = pd.to_datetime(train_for_timegan[date_col], errors='coerce')
if train_for_timegan[date_col].isna().all():
    train_for_timegan[date_col] = pd.Timestamp('2000-01-01')
    print(f"'{date_col}' is fully NaT after parsing, fallback to constant timestamp")
else:
    fill_value = train_for_timegan[date_col].dropna().min()
    train_for_timegan[date_col] = train_for_timegan[date_col].fillna(fill_value)

# Keep only features available in current train schema
TIMEGAN_FEATURE_COLUMNS = [c for c in (FEATURE_NUMERIC + FEATURE_CATEGORICAL) if c in train_for_timegan.columns]
TIMEGAN_NUMERIC_COLUMNS = [c for c in FEATURE_NUMERIC if c in TIMEGAN_FEATURE_COLUMNS]
TIMEGAN_CATEGORICAL_COLUMNS = [c for c in FEATURE_CATEGORICAL if c in TIMEGAN_FEATURE_COLUMNS]

if len(TIMEGAN_FEATURE_COLUMNS) == 0:
    raise ValueError("No feature columns available for TimeGAN sequence modeling")

# TimeGAN uses sigmoid output; normalize all feature columns to [0, 1]
TIMEGAN_SCALER = {}
timegan_norm_frame = train_for_timegan.copy()

for col in TIMEGAN_FEATURE_COLUMNS:
    col_series = pd.to_numeric(timegan_norm_frame[col], errors='coerce').fillna(0.0)
    col_min = float(col_series.min())
    col_max = float(col_series.max())

    if not np.isfinite(col_min) or not np.isfinite(col_max):
        col_min, col_max = 0.0, 1.0

    if abs(col_max - col_min) < 1e-12:
        norm_vals = np.zeros(len(col_series), dtype=float)
    else:
        norm_vals = (col_series - col_min) / (col_max - col_min)

    timegan_norm_frame[col] = np.clip(norm_vals, 0.0, 1.0)
    TIMEGAN_SCALER[col] = {'min': col_min, 'max': col_max}

def build_entity_windows(
    df,
    feature_cols,
    target_col,
    entity_col,
    date_col,
    seq_len,
    stride,
    min_history,
    allow_padded=False
):
    windows = []
    meta_records = []

    grouped = df.groupby(entity_col, dropna=False)
    n_entities_total = 0
    n_entities_used = 0
    n_padded = 0

    for entity_value, group_df in grouped:
        n_entities_total += 1
        grp = group_df.sort_values(date_col).reset_index(drop=True)
        history_len = len(grp)

        if history_len < min_history:
            continue

        if history_len >= seq_len:
            n_entities_used += 1
            start_points = range(0, history_len - seq_len + 1, stride)
            for start_idx in start_points:
                chunk = grp.iloc[start_idx:start_idx + seq_len].copy()
                windows.append(chunk[feature_cols].to_numpy(dtype=float))

                label_mode = pd.Series(chunk[target_col]).mode(dropna=True)
                label_value = int(label_mode.iloc[0]) if len(label_mode) > 0 else int(chunk[target_col].iloc[-1])

                meta_records.append({
                    'entity': str(entity_value),
                    'start_idx': int(start_idx),
                    'end_idx': int(start_idx + seq_len - 1),
                    'start_date': pd.to_datetime(chunk[date_col].iloc[0]),
                    'end_date': pd.to_datetime(chunk[date_col].iloc[-1]),
                    'window_label': label_value,
                    'is_padded': 0
                })
        elif allow_padded:
            # Optional padding for short-but-usable history
            n_entities_used += 1
            n_padded += 1
            chunk = grp.copy()
            pad_count = seq_len - history_len
            pad_rows = pd.concat([chunk.iloc[[-1]].copy() for _ in range(pad_count)], ignore_index=True)
            padded = pd.concat([chunk, pad_rows], ignore_index=True)
            windows.append(padded[feature_cols].to_numpy(dtype=float))

            label_mode = pd.Series(padded[target_col]).mode(dropna=True)
            label_value = int(label_mode.iloc[0]) if len(label_mode) > 0 else int(padded[target_col].iloc[-1])

            meta_records.append({
                'entity': str(entity_value),
                'start_idx': 0,
                'end_idx': int(seq_len - 1),
                'start_date': pd.to_datetime(padded[date_col].iloc[0]),
                'end_date': pd.to_datetime(padded[date_col].iloc[-1]),
                'window_label': label_value,
                'is_padded': 1
            })

    if len(windows) > 0:
        windows_array = np.asarray(windows, dtype=np.float32)
    else:
        windows_array = np.empty((0, seq_len, len(feature_cols)), dtype=np.float32)

    meta_df = pd.DataFrame(meta_records)
    stats_dict = {
        'entities_total': int(n_entities_total),
        'entities_used': int(n_entities_used),
        'entities_skipped': int(max(0, n_entities_total - n_entities_used)),
        'padded_windows': int(n_padded),
        'allow_padded_windows': bool(allow_padded)
    }
    return windows_array, meta_df, stats_dict

TIMEGAN_SEQUENCE_WINDOWS, TIMEGAN_SEQUENCE_META, TIMEGAN_WINDOW_BUILD_STATS = build_entity_windows(
    df=timegan_norm_frame,
    feature_cols=TIMEGAN_FEATURE_COLUMNS,
    target_col=target_col,
    entity_col=entity_col,
    date_col=date_col,
    seq_len=int(CONFIG['seq_len']),
    stride=max(1, int(CONFIG['sequence_stride'])),
    min_history=max(1, int(CONFIG['min_history_for_company'])),
    allow_padded=bool(CONFIG.get('allow_padded_windows', False))
)

if TIMEGAN_SEQUENCE_WINDOWS.shape[0] == 0:
    raise ValueError(
        "No sequence windows were created for TimeGAN. "
        "Check entity/date columns, seq_len, min_history_for_company, and allow_padded_windows."
    )

SEQUENCE_CLASS_COUNTS = TIMEGAN_SEQUENCE_META['window_label'].value_counts().sort_index() if len(TIMEGAN_SEQUENCE_META) > 0 else pd.Series(dtype=int)

print("TimeGAN sequence data prepared")
print(f"Entities: {timegan_norm_frame[entity_col].nunique()}")
print(f"Feature columns for TimeGAN: {len(TIMEGAN_FEATURE_COLUMNS)}")
print(f"Numeric features: {len(TIMEGAN_NUMERIC_COLUMNS)}")
print(f"Categorical features: {len(TIMEGAN_CATEGORICAL_COLUMNS)}")
print(f"Sequence windows: {TIMEGAN_SEQUENCE_WINDOWS.shape[0]}")
print(f"Window shape: {TIMEGAN_SEQUENCE_WINDOWS.shape}")
print(f"Classes in windows: {SEQUENCE_CLASS_COUNTS.to_dict()}")
print(f"Window builder stats: {TIMEGAN_WINDOW_BUILD_STATS}")

## 6. Define TimeGAN Utilities

Prepare helper utilities for scaling, inverse-scaling, and safe sequence sampling.

In [ ]:
# Utility helpers for TimeGAN pipeline
def inverse_timegan_scale(sequence_array, scaler_dict, feature_cols):
    """Inverse min-max scaling from [0, 1] back to train feature ranges."""
    arr = np.asarray(sequence_array, dtype=float).copy()
    for col_idx, col in enumerate(feature_cols):
        col_min = float(scaler_dict[col]['min'])
        col_max = float(scaler_dict[col]['max'])
        span = col_max - col_min
        if abs(span) < 1e-12:
            arr[:, :, col_idx] = col_min
        else:
            arr[:, :, col_idx] = arr[:, :, col_idx] * span + col_min
    return arr

def safe_timegan_sample(model, n_samples):
    """Safely generate `n_samples` sequence windows from a trained TimeGAN model."""
    raw_samples = model.generate(int(n_samples))
    arr = np.asarray(raw_samples, dtype=float)

    if arr.ndim == 2:
        arr = arr.reshape(1, arr.shape[0], arr.shape[1])

    if arr.ndim != 3:
        return np.empty((0, int(CONFIG['seq_len']), len(TIMEGAN_FEATURE_COLUMNS)), dtype=float)

    return arr

def postprocess_synthetic_rows(df, numeric_cols, categorical_cols):
    """Clip/round generated rows to keep schema compatible with preprocessed train data."""
    out = df.copy()

    for col in numeric_cols:
        if col in out.columns and col in train.columns:
            low = float(train[col].quantile(0.001))
            high = float(train[col].quantile(0.999))
            out[col] = pd.to_numeric(out[col], errors='coerce').clip(low, high)

    for col in categorical_cols:
        if col in out.columns and col in train.columns:
            if pd.api.types.is_integer_dtype(train[col]):
                cmin = int(train[col].min())
                cmax = int(train[col].max())
                out[col] = pd.to_numeric(out[col], errors='coerce').round().clip(cmin, cmax).fillna(cmin).astype(int)
            else:
                out[col] = out[col].astype(str)

    return out

print("TimeGAN utility helpers defined")

## 7. Train TimeGAN Models

In [8]:
if not TIMEGAN_AVAILABLE:
    raise ImportError(f"TimeGAN import failed: {TIMEGAN_IMPORT_ERROR}")

def build_timegan_model(batch_size_override=None):
    """Create tsgm TimeGAN from CONFIG."""
    return TimeGAN(
        seq_len=int(CONFIG['seq_len']),
        module=str(CONFIG.get('timegan_module', 'gru')),
        hidden_dim=int(CONFIG.get('timegan_latent_dim', 24)),
        n_features=len(TIMEGAN_FEATURE_COLUMNS),
        n_layers=int(CONFIG.get('timegan_n_layers', 3)),
        batch_size=int(batch_size_override if batch_size_override is not None else CONFIG['timegan_batch_size']),
        gamma=float(CONFIG['timegan_gamma'])
    )


def _set_all_seeds(seed):
    np.random.seed(int(seed))
    torch.manual_seed(int(seed))
    try:
        import tensorflow as _tf
        _tf.random.set_seed(int(seed))
    except Exception:
        pass


def _timegan_stability_score(model, reference_windows, n_eval=64):
    """Lower is better. Compare generated windows vs reference windows on mean/std stats."""
    ref = np.asarray(reference_windows, dtype=np.float32)
    if ref.ndim != 3 or ref.shape[0] == 0:
        return float('inf')

    n = int(min(max(8, n_eval), ref.shape[0]))
    gen = safe_timegan_sample(model, n)
    if gen.ndim != 3 or gen.shape[0] == 0:
        return float('inf')

    gen = gen[:, :ref.shape[1], :ref.shape[2]]
    ref_slice = ref[:gen.shape[0], :gen.shape[1], :gen.shape[2]]

    ref_mean = np.mean(ref_slice, axis=(0, 1))
    ref_std = np.std(ref_slice, axis=(0, 1))
    gen_mean = np.mean(gen, axis=(0, 1))
    gen_std = np.std(gen, axis=(0, 1))

    mean_gap = float(np.mean(np.abs(gen_mean - ref_mean)))
    std_gap = float(np.mean(np.abs(gen_std - ref_std)))
    return mean_gap + std_gap


def train_timegan_on_windows(window_data):
    """Train TimeGAN on sequence windows with optional multi-seed stability selection."""
    if len(window_data) == 0:
        raise ValueError("Empty window data")

    window_data = np.asarray(window_data, dtype=np.float32)
    max_windows = int(CONFIG.get('timegan_max_windows_per_model', 900))
    if window_data.shape[0] > max_windows:
        rng_sub = np.random.default_rng(int(CONFIG.get('random_seed', 42)))
        pick_idx = rng_sub.choice(window_data.shape[0], size=max_windows, replace=False)
        window_data = window_data[pick_idx]
        print(f"    Subsample windows for training: {len(pick_idx)}")

    stability_enabled = bool(CONFIG.get('timegan_stability_multi_seed', True))
    base_seed = int(CONFIG.get('random_seed', 42))
    candidate_seeds = CONFIG.get('timegan_candidate_seeds', [base_seed])
    candidate_seeds = [int(s) for s in candidate_seeds]
    if len(candidate_seeds) == 0:
        candidate_seeds = [base_seed]

    batch_fallback = [int(v) for v in CONFIG.get('timegan_batch_fallback', [int(CONFIG['timegan_batch_size']), 16, 8]) if int(v) > 0]
    batch_fallback = sorted(set(batch_fallback), reverse=True)

    if not stability_enabled:
        candidate_seeds = [candidate_seeds[0]]

    best_model = None
    best_score = float('inf')
    best_seed = None
    run_logs = []

    for seed in candidate_seeds:
        try:
            _set_all_seeds(seed)
            try:
                import tensorflow as _tf
                _tf.keras.backend.clear_session()
            except Exception:
                pass
            fit_ok = False
            last_err = None
            for bsz in batch_fallback:
                try:
                    m = build_timegan_model(batch_size_override=bsz)
                    m.compile()
                    m.fit(
                        data=window_data,
                        epochs=int(CONFIG['timegan_train_steps'])
                    )
                    fit_ok = True
                    if bsz != int(CONFIG['timegan_batch_size']):
                        print(f"    Memory fallback batch_size={bsz}")
                    break
                except Exception as fit_e:
                    last_err = fit_e
                    msg = str(fit_e).lower()
                    if ('out of memory' in msg) or ('resourceexhausted' in msg) or ('oom' in msg) or ('allocate' in msg):
                        continue
                    raise
            if not fit_ok:
                raise RuntimeError(f"TimeGAN fit failed after batch fallback: {last_err}")

            score = _timegan_stability_score(
                m,
                window_data,
                n_eval=int(CONFIG.get('timegan_eval_windows', 64))
            )
            run_logs.append({'seed': int(seed), 'stability_score': float(score)})

            if score < best_score:
                best_score = score
                best_seed = int(seed)
                best_model = m

            print(f"    seed={seed} stability_score={score:.6f}")
        except Exception as e:
            run_logs.append({'seed': int(seed), 'error': str(e)})
            print(f"    seed={seed} FAILED: {e}")

    if best_model is None:
        raise RuntimeError("All candidate TimeGAN seeds failed")

    best_model._selected_seed = int(best_seed)
    best_model._stability_score = float(best_score)
    best_model._stability_runs = run_logs
    return best_model


print("=== Training TimeGAN Models v3 (Stability Multi-Seed + Sector-Conditional) ===")
min_w = int(CONFIG.get('min_windows_per_class_for_timegan', 10))
priority_cls_set = set(int(c) for c in CONFIG.get('priority_class_labels', []))

TIMEGAN_MODELS = {}
TIMEGAN_TRAINING_REGISTRY = {
    'global': {}, 'per_class': {}, 'per_sector': {},
    'train_steps': int(CONFIG['timegan_train_steps']),
    'seq_len': int(CONFIG['seq_len']),
    'feature_columns': TIMEGAN_FEATURE_COLUMNS,
    'window_count': int(TIMEGAN_SEQUENCE_WINDOWS.shape[0]),
    'stability_multi_seed': bool(CONFIG.get('timegan_stability_multi_seed', True)),
    'candidate_seeds': [int(s) for s in CONFIG.get('timegan_candidate_seeds', [CONFIG.get('random_seed', 42)])]
}
SECTOR_FALLBACK_MAP = {}  # {class_label: model_key}

# â”€â”€ Global fallback: all priority class windows â”€â”€
prio_mask = TIMEGAN_SEQUENCE_META['window_label'].isin(priority_cls_set)
prio_wins = TIMEGAN_SEQUENCE_WINDOWS[prio_mask]
print(f"\n[Global] Training fallback on {len(prio_wins)} priority windows...")
try:
    TIMEGAN_MODELS['global'] = train_timegan_on_windows(prio_wins)
    TIMEGAN_TRAINING_REGISTRY['global'] = {
        'status': 'trained',
        'n': int(len(prio_wins)),
        'selected_seed': int(getattr(TIMEGAN_MODELS['global'], '_selected_seed', CONFIG.get('random_seed', 42))),
        'stability_score': float(getattr(TIMEGAN_MODELS['global'], '_stability_score', np.nan))
    }
    print("[Global] Training complete")
except Exception as e:
    TIMEGAN_TRAINING_REGISTRY['global'] = {'status': 'failed', 'error': str(e)}
    print(f"[Global] FAILED: {e}")

# â”€â”€ Sector-conditional fallback â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
sec_col = CONFIG.get('sector_column', 'sector')
if CONFIG.get('enable_sector_conditional', True) and sec_col in train.columns:
    print("\n[Sector-Conditional] Building sector-level fallback models...")
    ent_col = CONFIG['entity_column']
    if ent_col in train.columns:
        ent_sec = (
            train.groupby(ent_col)[sec_col]
            .agg(lambda x: x.mode()[0] if len(x) > 0 else 'Unknown')
        )
        TIMEGAN_SEQUENCE_META['sector'] = (
            TIMEGAN_SEQUENCE_META['entity'].map(ent_sec).fillna('Unknown')
        )
    else:
        TIMEGAN_SEQUENCE_META['sector'] = 'Unknown'

    for sec in TIMEGAN_SEQUENCE_META['sector'].unique():
        sec_mask = TIMEGAN_SEQUENCE_META['sector'] == sec
        sec_wins = TIMEGAN_SEQUENCE_WINDOWS[sec_mask.to_numpy()]
        if len(sec_wins) < max(10, min_w):
            TIMEGAN_TRAINING_REGISTRY['per_sector'][sec] = {
                'status': 'skipped', 'n': len(sec_wins)
            }
            continue

        safe_key = 'sector_' + re.sub(r'[^a-zA-Z0-9_]', '_', str(sec))
        try:
            TIMEGAN_MODELS[safe_key] = train_timegan_on_windows(sec_wins)
            TIMEGAN_TRAINING_REGISTRY['per_sector'][sec] = {
                'status': 'trained',
                'n': len(sec_wins),
                'selected_seed': int(getattr(TIMEGAN_MODELS[safe_key], '_selected_seed', CONFIG.get('random_seed', 42))),
                'stability_score': float(getattr(TIMEGAN_MODELS[safe_key], '_stability_score', np.nan))
            }
            print(f"  [Sector={sec}] OK ({len(sec_wins)} windows)")
        except Exception as e:
            TIMEGAN_TRAINING_REGISTRY['per_sector'][sec] = {
                'status': 'failed', 'error': str(e)
            }
            print(f"  [Sector={sec}] FAILED: {e}")

    # Map class -> dominant sector's model
    if ent_col in train.columns:
        cls_sec = (
            train.groupby([CONFIG['target_column'], sec_col])
            .size().reset_index(name='n')
            .sort_values('n', ascending=False)
            .groupby(CONFIG['target_column']).first()[sec_col]
        )
        for clabel, sname in cls_sec.items():
            sk = 'sector_' + re.sub(r'[^a-zA-Z0-9_]', '_', str(sname))
            if sk in TIMEGAN_MODELS:
                SECTOR_FALLBACK_MAP[int(clabel)] = sk
    print(f"  Sector fallback map: {SECTOR_FALLBACK_MAP}")
else:
    print("[Sector-Conditional] Disabled or sector column absent.")

# â”€â”€ Per-class TimeGAN (with min_windows guard) â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
print(f"\n[Per-Class] Dedicated models (min_windows={min_w})...")
for cls in sorted(priority_cls_set):
    cls_mask = TIMEGAN_SEQUENCE_META['window_label'].to_numpy() == int(cls)
    cls_wins = TIMEGAN_SEQUENCE_WINDOWS[cls_mask]
    n_ow = len(cls_wins)
    entry = {'n': n_ow, 'status': 'not_started'}

    if n_ow < min_w:
        fb = 'sector' if cls in SECTOR_FALLBACK_MAP else 'global'
        entry['status'] = 'skipped_insufficient'
        entry['reason'] = f'{n_ow} < {min_w}'
        print(f"  [Cls {cls}] SKIP ({n_ow} wins < {min_w}) -> {fb} fallback")
    else:
        try:
            print(f"  [Cls {cls}] Training on {n_ow} windows...")
            TIMEGAN_MODELS[f'class_{cls}'] = train_timegan_on_windows(cls_wins)
            try:
                TIMEGAN_MODELS[f'class_{cls}'].save(
                    str(MODELS_DIR / f'timegan_class_{cls}.pkl')
                )
                entry['save'] = 'ok'
            except Exception as se:
                entry['save'] = str(se)

            entry['status'] = 'trained'
            entry['selected_seed'] = int(getattr(TIMEGAN_MODELS[f'class_{cls}'], '_selected_seed', CONFIG.get('random_seed', 42)))
            entry['stability_score'] = float(getattr(TIMEGAN_MODELS[f'class_{cls}'], '_stability_score', np.nan))
            print(f"  [Cls {cls}] OK")
        except Exception as e:
            entry.update({'status': 'failed', 'error': str(e)})
            print(f"  [Cls {cls}] FAILED: {e}")

    TIMEGAN_TRAINING_REGISTRY['per_class'][str(cls)] = entry

trained_cls = [k for k, v in TIMEGAN_TRAINING_REGISTRY['per_class'].items()
               if v.get('status') == 'trained']
skipped_cls = [k for k, v in TIMEGAN_TRAINING_REGISTRY['per_class'].items()
               if 'skipped' in v.get('status', '')]
print(f"\nPer-class trained: {trained_cls}")
print(f"Per-class skipped: {skipped_cls}")
print(f"Sector fallbacks:  {list(SECTOR_FALLBACK_MAP.items())}")


ImportError: TimeGAN import failed: No module named 'tsgm'

In [ ]:
# Save TimeGAN training registry
def to_json_compatible(obj):
    """Recursively convert objects to JSON-safe Python types."""
    if isinstance(obj, dict):
        converted = {}
        for key, value in obj.items():
            if isinstance(key, np.generic):
                safe_key = key.item()
            elif isinstance(key, (str, int, float, bool)) or key is None:
                safe_key = key
            else:
                safe_key = str(key)

            if not isinstance(safe_key, (str, int, float, bool)) and safe_key is not None:
                safe_key = str(safe_key)

            converted[safe_key] = to_json_compatible(value)
        return converted

    if isinstance(obj, (list, tuple)):
        return [to_json_compatible(item) for item in obj]

    if isinstance(obj, np.generic):
        return to_json_compatible(obj.item())

    if isinstance(obj, float):
        return obj if np.isfinite(obj) else None

    if isinstance(obj, pd.Timestamp):
        return obj.isoformat()

    return obj

registry_path = MODELS_DIR / 'timegan_training_registry.json'
registry_payload = to_json_compatible(TIMEGAN_TRAINING_REGISTRY)
with open(registry_path, 'w', encoding='utf-8') as f:
    json.dump(registry_payload, f, indent=2, ensure_ascii=False, default=str)

print(f" Training registry saved to {registry_path}")

## 8. Generate Synthetic Samples

**Strategy:** Minority-focused sequence generation with per-class TimeGAN and global fallback

In [ ]:
# Sampling Plan v3 â€” Log-Ratio + Validation-Proxy Budget Optimization
# Changes vs v2:
# - Add budget optimization across candidate synthetic ratios
# - Use quick val-proxy objective (macro-F1 + balanced accuracy)
# - Keep Hamilton allocation and class-priority logic

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, balanced_accuracy_score

full_class_range = list(range(int(CONFIG['target_min_label']),
                               int(CONFIG['target_max_label']) + 1))
class_counts = (
    train[CONFIG['target_column']].value_counts()
    .reindex(full_class_range, fill_value=0).sort_index().astype(int)
)
nonzero_counts = class_counts[class_counts > 0]
median_count = float(nonzero_counts.median()) if len(nonzero_counts) > 0 else 1.0
p75_count = float(nonzero_counts.quantile(0.75)) if len(nonzero_counts) > 0 else 1.0

print("=== Sampling Plan v3 (Budget Optimization Enabled) ===")
print(class_counts)
print(f"\nMedian={median_count:.0f}  P75={p75_count:.0f}")

priority_cls_list = [int(c) for c in CONFIG.get('priority_class_labels', [])]
priority_cls_present = [c for c in priority_cls_list
                        if c in class_counts.index and class_counts[c] > 0]

strategy = CONFIG.get('balance_strategy', 'log_ratio')
prio_boost = float(CONFIG.get('priority_boost_multiplier', 1.5))
prio_min = int(CONFIG.get('priority_min_count', 300))
prio_maxf = float(CONFIG.get('priority_max_oversample_factor', 150.0))
distressed_label = int(CONFIG.get('distressed_label', 0))
distressed_target_ratio = float(CONFIG.get('distressed_target_ratio_vs_majority', 0.85))
distressed_min_synth = int(CONFIG.get('distressed_min_synthetic', 700))
majority_count = int(nonzero_counts.max()) if len(nonzero_counts) > 0 else 0

print(f"\nBalance strategy: '{strategy}'")
print(f"Priority classes present: {priority_cls_present}")


def compute_target(cls_label: int, real_cnt: int) -> int:
    """Compute augmentation target for a class with Distressed-focused floor."""
    cls_label = int(cls_label)
    is_p = cls_label in priority_cls_list
    is_distressed = cls_label == distressed_label
    cnt = max(real_cnt, 0)

    if strategy == 'log_ratio':
        base = int(np.sqrt(cnt * p75_count)) if cnt > 0 else int(p75_count)
        if not is_p:
            base = min(base, int(p75_count))
    elif strategy == 'sqrt_flat':
        base = int(np.sqrt(cnt * median_count)) if cnt > 0 else int(median_count)
    else:
        base = int(median_count)

    if is_p:
        boosted = max(int(base * prio_boost), prio_min)
        cap = int(np.ceil(cnt * prio_maxf)) if cnt > 0 else boosted
        base = min(boosted, cap)
    elif cnt == 0:
        base = 0

    if is_distressed:
        distressed_floor = int(np.ceil(majority_count * distressed_target_ratio))
        distressed_floor = max(distressed_floor, cnt + distressed_min_synth)
        base = max(base, distressed_floor)

    return max(0, base)


def allocate_budget(need_map: dict, budget: int) -> dict:
    """Allocate synthetic budget proportionally (Hamilton method)."""
    if budget <= 0 or sum(need_map.values()) <= 0:
        return {k: 0 for k in need_map}
    total = sum(need_map.values())
    if total <= budget:
        return need_map.copy()

    scaled = {}
    rems = []
    alloc = 0
    for k, v in need_map.items():
        ex = budget * (v / total)
        fl = int(np.floor(ex))
        scaled[k] = fl
        alloc += fl
        rems.append((ex - fl, k))

    for _, k in sorted(rems, reverse=True)[:budget - alloc]:
        scaled[k] += 1
    return scaled


raw_need = {
    int(c): max(0, compute_target(int(c), int(v)) - int(v))
    for c, v in class_counts.items()
}


def build_plan_for_ratio(ratio_value: float):
    max_synth_local = int(len(train) * float(ratio_value))
    alloc = allocate_budget(raw_need, max_synth_local)
    total = int(sum(alloc.values()))
    expected = class_counts.add(pd.Series(alloc), fill_value=0).astype(int)
    return alloc, total, max_synth_local, expected


selected_ratio = float(CONFIG.get('max_synthetic_ratio', 0.60))
budget_eval_rows = []
budget_opt_enabled = bool(CONFIG.get('budget_optimization_enabled', False))
ratio_candidates = [float(r) for r in CONFIG.get('budget_ratio_candidates', [selected_ratio])]
if selected_ratio not in ratio_candidates:
    ratio_candidates.append(selected_ratio)
ratio_candidates = sorted(set([r for r in ratio_candidates if r > 0]))

if budget_opt_enabled and len(ratio_candidates) > 1 and len(val) > 0:
    print("\n=== Budget Optimization (validation proxy) ===")

    feat_cols = [c for c in TIMEGAN_FEATURE_COLUMNS if c in train.columns and c in val.columns]
    can_eval = (
        len(feat_cols) > 0
        and train[CONFIG['target_column']].nunique() > 1
        and val[CONFIG['target_column']].nunique() > 0
        and str(CONFIG.get('budget_optimization_mode', 'val_proxy')).lower() == 'val_proxy'
    )

    if can_eval:
        w_f1 = float(CONFIG.get('budget_objective', {}).get('macro_f1_weight', 0.7))
        w_bal = float(CONFIG.get('budget_objective', {}).get('balanced_accuracy_weight', 0.3))

        for r in ratio_candidates:
            alloc_r, total_r, max_synth_r, expected_r = build_plan_for_ratio(r)

            # Class-level reweighting proxy: mimic augmented target counts without generating synthetic rows
            cls_weight_map = {}
            for cls_label in class_counts.index:
                real_cnt = int(class_counts.get(cls_label, 0))
                exp_cnt = int(expected_r.get(cls_label, real_cnt))
                cls_weight_map[int(cls_label)] = float(exp_cnt / max(real_cnt, 1))

            sample_weight = train[CONFIG['target_column']].map(lambda x: cls_weight_map.get(int(x), 1.0)).astype(float).to_numpy()

            clf = RandomForestClassifier(
                n_estimators=int(CONFIG.get('rf_proxy_estimators', 120)),
                random_state=RANDOM_SEED,
                n_jobs=int(CONFIG.get('rf_n_jobs', 2)),
                class_weight='balanced_subsample',
                min_samples_leaf=2
            )
            clf.fit(train[feat_cols].fillna(0), train[CONFIG['target_column']], sample_weight=sample_weight)
            pred = clf.predict(val[feat_cols].fillna(0))

            macro_f1 = float(f1_score(val[CONFIG['target_column']], pred, average='macro'))
            bal_acc = float(balanced_accuracy_score(val[CONFIG['target_column']], pred))
            score = float(w_f1 * macro_f1 + w_bal * bal_acc)

            budget_eval_rows.append({
                'ratio': float(r),
                'max_synth_budget': int(max_synth_r),
                'planned_synth_rows': int(total_r),
                'val_macro_f1_proxy': macro_f1,
                'val_balanced_acc_proxy': bal_acc,
                'proxy_score': score
            })

        best_row = max(budget_eval_rows, key=lambda x: x['proxy_score'])
        selected_ratio = float(best_row['ratio'])
        print(f"Selected synthetic ratio by proxy: {selected_ratio:.2f}")
        print("Candidates:")
        for row in budget_eval_rows:
            print(
                f"  ratio={row['ratio']:.2f} | score={row['proxy_score']:.4f} "
                f"(f1={row['val_macro_f1_proxy']:.4f}, bal_acc={row['val_balanced_acc_proxy']:.4f})"
            )
    else:
        print("Budget optimization skipped: insufficient features/labels or mode disabled")

samples_per_class, total_synth, max_synth, expected_counts = build_plan_for_ratio(selected_ratio)
windows_per_class = {
    int(c): int(np.ceil(n / max(1, CONFIG['seq_len'])))
    for c, n in samples_per_class.items() if int(n) > 0
}

TIMEGAN_BUDGET_OPTIMIZATION_LOG = {
    'enabled': bool(budget_opt_enabled),
    'mode': str(CONFIG.get('budget_optimization_mode', 'none')),
    'selected_ratio': float(selected_ratio),
    'candidates': budget_eval_rows
}

print(f"\nBudget: {max_synth} | Planned: {total_synth} ({total_synth/len(train):.1%} of train)")
print(f"\n{'Class':>6} {'Real':>7} {'Target':>8} {'Need':>8} {'Windows':>9}  Model-Source")
print("-" * 65)
for cls in sorted(samples_per_class):
    nd = samples_per_class[cls]
    if nd <= 0:
        continue
    rl = int(class_counts.get(cls, 0))
    tg = compute_target(int(cls), rl)
    wn = windows_per_class.get(cls, 0)
    mk = f'class_{cls}'
    sk = SECTOR_FALLBACK_MAP.get(int(cls), '')
    if mk in TIMEGAN_MODELS:
        src = 'per-class'
    elif sk and sk in TIMEGAN_MODELS:
        src = f'sector({sk})'
    elif 'global' in TIMEGAN_MODELS:
        src = 'global'
    else:
        src = 'NONE !'
    print(f"{cls:>6} {rl:>7} {tg:>8} {nd:>8} {wn:>9}  {src}")

print(f"\nExpected class counts after augmentation:")
print(expected_counts.sort_index())

nec = expected_counts[expected_counts > 0]
br = (nonzero_counts.min() / nonzero_counts.max()
      if len(nonzero_counts) > 0 and nonzero_counts.max() > 0 else float('nan'))
ar = (nec.min() / nec.max()
      if len(nec) > 0 and nec.max() > 0 else float('nan'))
print(f"\nImbalance ratio before: {br:.4f}" if np.isfinite(br) else "\nImbalance before: N/A")
print(f"Expected ratio after:   {ar:.4f}" if np.isfinite(ar) else "Expected after:  N/A")

In [ ]:
# Generation v4: strict synthetic ticker length for downstream window eligibility
#
# Priority model source:
#   1. per-class TimeGAN
#   2. sector-conditional TimeGAN
#   3. global TimeGAN
#   4. Ordinal-aware MixUp fallback
#
# Key fix:
#   - Every synthetic ticker is materialized with at least min required records
#   - Prevent synthetic tickers shorter than downstream INPUT_SIZE + HORIZON
#   - Add relaxed acceptance fallback when strict JS filter rejects all windows

from sklearn.ensemble import RandomForestClassifier
from scipy.spatial.distance import jensenshannon

# -- [Step 1] RF Quality Classifier -------------------------------------------
print("=== [1] RF Quality Classifier ===")
quality_clf = RandomForestClassifier(
    n_estimators=int(CONFIG.get('rf_quality_estimators', 120)), random_state=RANDOM_SEED, n_jobs=int(CONFIG.get('rf_n_jobs', 2)),
    class_weight='balanced', max_depth=15, min_samples_leaf=2
)
quality_clf.fit(
    train[TIMEGAN_FEATURE_COLUMNS].fillna(0),
    train[CONFIG['target_column']]
)
print(f"RF trained on {len(train)} samples")

# -- [Step 2] Per-class reference distributions for JS divergence ------------
print("\n[2] Building reference distributions for JS filter...")
bins_cfg = {}
for col in TIMEGAN_NUMERIC_COLUMNS:
    lo = float(train[col].quantile(0.01))
    hi = float(train[col].quantile(0.99))
    bins_cfg[col] = np.linspace(lo - 1e-9, hi + 1e-9, 31)

REF_DISTS = {}
for lbl in sorted(train[CONFIG['target_column']].unique()):
    cd = train[train[CONFIG['target_column']] == lbl]
    ch = {}
    for col, bins in bins_cfg.items():
        if col in cd.columns:
            h, _ = np.histogram(cd[col].dropna(), bins=bins, density=True)
            h = h + 1e-8
            h /= h.sum()
            ch[col] = h
    REF_DISTS[int(lbl)] = ch
print(f"Reference distributions built for {len(REF_DISTS)} classes")


def js_score_window(win_df: pd.DataFrame, cls: int) -> float:
    """Average JS divergence between a synthetic window and class reference."""
    if int(cls) not in REF_DISTS:
        return 0.0
    ref = REF_DISTS[int(cls)]
    vals = []
    for col, rh in ref.items():
        if col not in win_df.columns:
            continue
        h, _ = np.histogram(win_df[col].dropna(), bins=bins_cfg[col], density=True)
        h = h + 1e-8
        h /= h.sum()
        vals.append(float(jensenshannon(rh, h)))
    return float(np.mean(vals)) if vals else 0.0


def ordinal_mixup(
    train_df: pd.DataFrame,
    cls: int,
    n_needed: int,
    feat_cols: list,
    tgt_col: str,
    alpha: float = 0.3,
    adjacent: bool = True,
    seed: int = 42
) -> pd.DataFrame:
    """
    Ordinal-aware MixUp: interpolate between class `cls` and adjacent classes.
    Only mix class i with i+/-1 to preserve ordinal semantics.
    """
    rng = np.random.default_rng(seed)
    cls_a = train_df[train_df[tgt_col] == cls][feat_cols].dropna()
    if len(cls_a) == 0:
        return pd.DataFrame()

    partners = []
    if adjacent:
        for adj in [cls - 1, cls + 1]:
            p = train_df[train_df[tgt_col] == adj][feat_cols].dropna()
            if len(p) > 0:
                partners.append(p)
    if not partners:
        partners.append(cls_a)

    rows = []
    for _ in range(n_needed):
        a = cls_a.sample(1, replace=True, random_state=None).values[0]
        b = partners[int(rng.integers(0, len(partners)))].sample(
            1, replace=True, random_state=None
        ).values[0]
        lam = float(rng.beta(alpha, alpha))
        rows.append(lam * a + (1.0 - lam) * b)

    out = pd.DataFrame(rows, columns=feat_cols)
    out[tgt_col] = cls
    return out


# -- [Step 3] Main Generation Loop -------------------------------------------
print("\n[3] Generating synthetic samples (TimeGAN + MixUp fallback)...")

tgt = CONFIG['target_column']
dtc = CONFIG['date_column']
ent = CONFIG['entity_column']
sl = int(CONFIG['seq_len'])
js_t_base = float(CONFIG.get('js_divergence_threshold', 0.20))
rf_t_base = float(CONFIG.get('rf_quality_threshold', 0.50))
use_mix = bool(CONFIG.get('enable_ordinal_mixup_fallback', True))
mix_a = float(CONFIG.get('mixup_alpha', 0.3))
mix_adj = bool(CONFIG.get('mixup_adjacent_only', True))
rng_main = np.random.default_rng(int(CONFIG['random_seed']))

def get_adaptive_thresholds(cls_label: int, real_c: int, base_js: float, base_rf: float):
    if int(cls_label) == int(CONFIG.get('distressed_label', 0)):
        return (
            base_js * float(CONFIG.get('distressed_js_relax_multiplier', 1.6)),
            base_rf * float(CONFIG.get('distressed_rf_relax_multiplier', 0.75)),
        )
    if real_c < 10:
        return base_js * 1.5, base_rf * 0.65
    elif real_c < 30:
        return base_js * 1.25, base_rf * 0.85
    else:
        return base_js, base_rf


def select_relaxed_windows(candidate_rows, max_windows):
    """Pick best rejected windows by lowest JS and highest RF match."""
    if max_windows <= 0 or len(candidate_rows) == 0:
        return []
    ordered = sorted(candidate_rows, key=lambda x: (x['js'], -x['rf_match']))
    return ordered[:max_windows]


min_hist_for_downstream = int(CONFIG.get('downstream_min_history_for_window', 9))
min_points_per_synth_ticker = int(CONFIG.get('synthetic_min_points_per_ticker', min_hist_for_downstream))
min_points_per_synth_ticker = max(min_points_per_synth_ticker, min_hist_for_downstream)
sequence_block_len = max(
    2,
    min_hist_for_downstream,
    min_points_per_synth_ticker,
    int(CONFIG.get('mixup_sequence_block_len', min_points_per_synth_ticker))
)
force_all_tickers_eligible = bool(CONFIG.get('downstream_enforce_all_synthetic_tickers_eligible', True))


def _refdt(series):
    d = pd.to_datetime(series, errors='coerce').dropna()
    return d.max() if len(d) > 0 else pd.Timestamp(datetime.now().date())


base_dt = _refdt(train[dtc]) if dtc in train.columns else pd.Timestamp(datetime.now().date())

# Enforce synthetic rating_date to stay within active train-fold date range
train_date_values = []
if dtc in train.columns:
    train_date_values = pd.to_datetime(train[dtc], errors='coerce').dropna().tolist()

if len(train_date_values) > 0:
    train_date_min = min(train_date_values)
    train_date_max = max(train_date_values)
else:
    train_date_min = pd.Timestamp(datetime.now().date())
    train_date_max = train_date_min


def clamp_to_train_date_range(ts):
    parsed = pd.to_datetime(ts, errors='coerce')
    if pd.isna(parsed):
        return train_date_max
    if parsed < train_date_min:
        return train_date_min
    if parsed > train_date_max:
        return train_date_max
    return parsed


# Build empirical date pools by class
class_date_pool = {}
class_step_pool = {}
max_step_days = int(CONFIG.get('max_date_step_days', 365))
if dtc in train.columns:
    for lbl, grp in train.groupby(tgt):
        dvals = pd.to_datetime(grp[dtc], errors='coerce').dropna().tolist()
        if len(dvals) > 0:
            class_date_pool[int(lbl)] = sorted(dvals)

        step_vals = []
        if ent in grp.columns:
            for _, eg in grp.groupby(ent):
                ds = pd.to_datetime(eg[dtc], errors='coerce').dropna().sort_values()
                if len(ds) > 1:
                    diffs = ds.diff().dt.days.dropna()
                    step_vals.extend([int(v) for v in diffs if 1 <= int(v) <= max_step_days])
        else:
            ds = pd.to_datetime(grp[dtc], errors='coerce').dropna().sort_values()
            if len(ds) > 1:
                diffs = ds.diff().dt.days.dropna()
                step_vals.extend([int(v) for v in diffs if 1 <= int(v) <= max_step_days])

        class_step_pool[int(lbl)] = step_vals if len(step_vals) > 0 else [90]

print(f"Train fold date range for synthetic data: [{train_date_min.date()} -> {train_date_max.date()}]")
print(f"Synthetic sequence block length: {sequence_block_len}")
print(f"Minimum points per synthetic ticker: {min_points_per_synth_ticker}")
print(f"Force all synthetic tickers eligible: {force_all_tickers_eligible}")


def oversample_factor(rc: int) -> float:
    """Extra generation multiplier to compensate for JS+RF filter losses."""
    if rc < 5:
        return 8.0
    if rc < 15:
        return 6.0
    if rc < 30:
        return 4.0
    if rc < 60:
        return 3.0
    return 2.0


def _pad_rows_to_min_length(rows, min_len):
    """Pad short row blocks to min_len by resampling + tiny numeric jitter."""
    if len(rows) == 0:
        return [], 0
    if len(rows) >= min_len:
        return rows, 0

    padded = [dict(r) for r in rows]
    added = 0
    while len(padded) < min_len:
        src = dict(padded[len(padded) % len(rows)])
        for c in TIMEGAN_NUMERIC_COLUMNS:
            if c in src:
                try:
                    src[c] = float(src[c]) + float(rng_main.normal(0.0, 1e-3))
                except Exception:
                    pass
        src['synthetic_tail_pad'] = 1
        padded.append(src)
        added += 1

    for i in range(min(len(rows), len(padded))):
        padded[i]['synthetic_tail_pad'] = 0

    return padded, int(added)


def materialize_rows_as_sequences(
    rows,
    cls_label,
    block_len,
    date_pool,
    step_pool,
    counter_start,
    force_min_ticker_len=True,
    min_ticker_len=9,
    prefix='SYN'
):
    """Assign ticker/date to synthetic rows in sequence blocks to ensure usable entity histories."""
    if len(rows) == 0:
        return [], counter_start, 0, 0, 0, 0

    working_rows = [dict(r) for r in rows]
    n_pad_added = 0

    if len(working_rows) < min_ticker_len and force_min_ticker_len:
        working_rows, n_pad_added = _pad_rows_to_min_length(working_rows, min_ticker_len)

    df_rows = pd.DataFrame(working_rows).reset_index(drop=True)
    n_rows = len(df_rows)

    if n_rows <= block_len:
        entity_sizes = [n_rows]
    else:
        n_entities = max(1, n_rows // block_len)
        base = n_rows // n_entities
        rem = n_rows % n_entities
        entity_sizes = [base + (1 if i < rem else 0) for i in range(n_entities)]

    if force_min_ticker_len:
        entity_sizes = [max(int(s), int(min_ticker_len)) for s in entity_sizes]

    out_rows = []
    row_ptr = 0
    counter = int(counter_start)
    date_clamped = 0
    short_entities = 0

    for size in entity_sizes:
        if size < min_hist_for_downstream:
            short_entities += 1

        ticker_value = f"{prefix}_{int(cls_label)}_{counter:07d}"
        counter += 1

        if len(date_pool) > 0:
            start_dt = date_pool[int(rng_main.integers(0, len(date_pool)))]
        else:
            start_dt = base_dt

        current_dt = clamp_to_train_date_range(start_dt)

        min_required_days = size
        if (train_date_max - current_dt).days < min_required_days:
            current_dt = train_date_max - pd.Timedelta(days=min_required_days)
            if current_dt < train_date_min:
                current_dt = train_date_min

        for local_idx in range(size):
            if row_ptr < n_rows:
                rec = df_rows.iloc[row_ptr].to_dict()
                row_ptr += 1
            else:
                src = df_rows.iloc[local_idx % n_rows].to_dict()
                rec = dict(src)
                for c in TIMEGAN_NUMERIC_COLUMNS:
                    if c in rec:
                        try:
                            rec[c] = float(rec[c]) + float(rng_main.normal(0.0, 1e-3))
                        except Exception:
                            pass
                rec['synthetic_tail_pad'] = 1
                n_pad_added += 1

            if ent in train.columns:
                rec[ent] = ticker_value

            if dtc in train.columns:
                bounded_dt = clamp_to_train_date_range(current_dt)
                if bounded_dt != current_dt:
                    date_clamped += 1
                rec[dtc] = bounded_dt

                step_days = int(step_pool[int(rng_main.integers(0, len(step_pool)))]) if len(step_pool) > 0 else 90

                remaining_steps = size - local_idx - 1
                if remaining_steps > 0:
                    max_step_allowed = (train_date_max - bounded_dt).days - remaining_steps
                    if step_days > max_step_allowed:
                        step_days = max(1, max_step_allowed)
                else:
                    step_days = 1

                current_dt = bounded_dt + pd.Timedelta(days=step_days)

            out_rows.append(rec)

    return out_rows, counter, date_clamped, int(len(entity_sizes)), int(short_entities), int(n_pad_added)


all_rows = []
gen_log = []
entity_counter = 0

date_clamp_total = 0
short_entity_total = 0
synthetic_entity_total = 0
total_padding_rows_added = 0

for cls, nw in sorted(windows_per_class.items()):
    if int(nw) <= 0:
        continue

    real_c = int(class_counts.get(int(cls), 0))
    need_r = int(samples_per_class.get(int(cls), 0))
    current_js_t, current_rf_t = get_adaptive_thresholds(int(cls), real_c, js_t_base, rf_t_base)

    # Model selection: per-class > sector > global
    mk = f'class_{int(cls)}'
    sk = SECTOR_FALLBACK_MAP.get(int(cls), '')
    if mk in TIMEGAN_MODELS:
        mdl = TIMEGAN_MODELS[mk]
        msrc = mk
    elif sk and sk in TIMEGAN_MODELS:
        mdl = TIMEGAN_MODELS[sk]
        msrc = sk
    elif 'global' in TIMEGAN_MODELS:
        mdl = TIMEGAN_MODELS['global']
        msrc = 'global'
    else:
        mdl = None
        msrc = None

    cls_timegan_rows = []
    accepted_windows = 0
    js_rejected = 0
    rf_rejected = 0
    relaxed_windows_accepted = 0

    if mdl is not None:
        try:
            ovf = oversample_factor(real_c)
            over_nw = int(np.ceil(nw * ovf))
            class_cap = int(CONFIG.get('timegan_generation_max_windows_per_class', 256))
            if int(cls) == int(CONFIG.get('distressed_label', 0)):
                class_cap = int(CONFIG.get('distressed_windows_cap', class_cap))
            over_nw = min(over_nw, class_cap)
            print(
                f"\nCls {cls} (real={real_c}, need_rows={need_r}): "
                f"request {over_nw} windows via {msrc} ({ovf}x)"
            )

            sn = safe_timegan_sample(mdl, over_nw)
            if sn.shape[0] == 0:
                print("  WARNING: model returned 0 windows")
            else:
                sn = sn[:, :sl, :len(TIMEGAN_FEATURE_COLUMNS)]
                sr = inverse_timegan_scale(sn, TIMEGAN_SCALER, TIMEGAN_FEATURE_COLUMNS)
                relaxed_candidates = []

                for wi in range(sr.shape[0]):
                    blk = sr[wi]
                    w_rows = []
                    for si in range(sl):
                        rd = {c: float(blk[si, ci]) for ci, c in enumerate(TIMEGAN_FEATURE_COLUMNS)}
                        rd[tgt] = int(cls)
                        rd['is_mixup'] = 0
                        rd['synthetic_source'] = 'timegan'
                        rd['synthetic_tail_pad'] = 0
                        w_rows.append(rd)

                    wdf = pd.DataFrame(w_rows)
                    preds = quality_clf.predict(wdf[TIMEGAN_FEATURE_COLUMNS].fillna(0))
                    rf_match = float(np.mean(preds == int(cls)))

                    # JS divergence filter
                    js_val = js_score_window(wdf, int(cls))
                    pass_js = js_val <= current_js_t
                    pass_rf = rf_match >= current_rf_t

                    if pass_js and pass_rf:
                        cls_timegan_rows.extend(w_rows)
                        accepted_windows += 1
                        if len(cls_timegan_rows) >= need_r:
                            break
                        continue

                    if not pass_js:
                        js_rejected += 1
                    elif not pass_rf:
                        rf_rejected += 1

                    relaxed_candidates.append({
                        'js': float(js_val),
                        'rf_match': float(rf_match),
                        'rows': w_rows,
                    })

                if len(cls_timegan_rows) < need_r and len(relaxed_candidates) > 0:
                    relaxed_gap_windows = int(np.ceil((need_r - len(cls_timegan_rows)) / max(1, sl)))
                    for cand in select_relaxed_windows(relaxed_candidates, relaxed_gap_windows):
                        for row in cand['rows']:
                            row_relaxed = dict(row)
                            row_relaxed['synthetic_source'] = 'timegan_relaxed'
                            cls_timegan_rows.append(row_relaxed)
                        relaxed_windows_accepted += 1
                        if len(cls_timegan_rows) >= need_r:
                            break

                print(
                    f"  Accepted windows={accepted_windows}  "
                    f"Relaxed={relaxed_windows_accepted}  "
                    f"JS-rejected={js_rejected}  RF-rejected={rf_rejected}"
                )
        except Exception as e:
            print(f"  TimeGAN FAILED: {e}")

    # Keep rows within class budget (can be expanded later for min ticker length)
    if len(cls_timegan_rows) > need_r:
        cls_timegan_rows = cls_timegan_rows[:need_r]

    # Ordinal MixUp fallback: fill remaining gap
    cls_mix_rows = []
    gap = need_r - len(cls_timegan_rows)
    if gap > 0 and use_mix:
        print(f"  -> MixUp fallback: {gap} rows needed for cls {cls}")
        mdf = ordinal_mixup(
            train_df=train,
            cls=int(cls),
            n_needed=gap,
            feat_cols=TIMEGAN_FEATURE_COLUMNS,
            tgt_col=tgt,
            alpha=mix_a,
            adjacent=mix_adj,
            seed=int(CONFIG['random_seed'])
        )
        if len(mdf) > 0:
            for _, row in mdf.iterrows():
                rec = row.to_dict()
                rec['is_mixup'] = 1
                rec['synthetic_source'] = 'mixup'
                rec['synthetic_tail_pad'] = 0
                cls_mix_rows.append(rec)
            print(f"  MixUp: created {len(cls_mix_rows)} rows")
    elif gap > 0:
        print(f"  WARNING: gap={gap} rows for cls {cls}, MixUp disabled")

    cls_all_rows = cls_timegan_rows + cls_mix_rows

    # Materialize class rows into sequence blocks (ticker/date assignment)
    dpool = class_date_pool.get(int(cls), train_date_values)
    spool = class_step_pool.get(int(cls), [90])
    mat_rows, entity_counter, clamped_n, n_entities, n_short, n_pad = materialize_rows_as_sequences(
        rows=cls_all_rows,
        cls_label=int(cls),
        block_len=sequence_block_len,
        date_pool=dpool,
        step_pool=spool,
        counter_start=entity_counter,
        force_min_ticker_len=force_all_tickers_eligible,
        min_ticker_len=min_points_per_synth_ticker,
        prefix='SYN'
    )

    all_rows.extend(mat_rows)
    date_clamp_total += int(clamped_n)
    synthetic_entity_total += int(n_entities)
    short_entity_total += int(n_short)
    total_padding_rows_added += int(n_pad)

    gen_log.append({
        'class_label': int(cls),
        'windows_requested': int(nw),
        'windows_generated': int(accepted_windows),
        'rows_generated': int(len(mat_rows)),
        'js_rejected': int(js_rejected),
        'rf_rejected': int(rf_rejected),
        'relaxed_windows_accepted': int(relaxed_windows_accepted),
        'model_source': str(msrc),
        'mixup_rows': int(len(cls_mix_rows)),
        'synthetic_entities': int(n_entities),
        'short_entities': int(n_short),
        'padding_rows_added': int(n_pad)
    })

# -- Combine all synthetic rows ------------------------------------------------
if len(all_rows) == 0:
    synthetic_df = pd.DataFrame()
    print("\nWARNING: 0 synthetic rows generated")
else:
    synthetic_df = pd.DataFrame(all_rows)
    if 'is_mixup' not in synthetic_df.columns:
        synthetic_df['is_mixup'] = 0
    synthetic_df['is_mixup'] = synthetic_df['is_mixup'].fillna(0).astype(int)

    if 'synthetic_tail_pad' not in synthetic_df.columns:
        synthetic_df['synthetic_tail_pad'] = 0
    synthetic_df['synthetic_tail_pad'] = synthetic_df['synthetic_tail_pad'].fillna(0).astype(int)

    print(f"\nTotal synthetic: {len(synthetic_df)}")
    print(f"  Date clamps (synthetic rows): {date_clamp_total}")
    print(f"  Synthetic entities: {synthetic_entity_total}")
    print(f"  Short synthetic entities (<{min_hist_for_downstream}): {short_entity_total}")
    print(f"  Padding rows added: {total_padding_rows_added}")
    print(f"\nClass distribution (synthetic):")
    print(synthetic_df[tgt].value_counts().sort_index())

# Downstream eligibility audit for synthetic rows
TIMEGAN_SYNTH_ENTITY_AUDIT = {}
if len(synthetic_df) > 0 and ent in synthetic_df.columns:
    ticker_counts = synthetic_df[ent].astype(str).value_counts()
    eligible_tickers = ticker_counts[ticker_counts >= min_hist_for_downstream].index.tolist()
    eligible_rows = int(synthetic_df[ent].astype(str).isin(eligible_tickers).sum())
    eligible_ratio = float(eligible_rows / max(len(synthetic_df), 1))

    TIMEGAN_SYNTH_ENTITY_AUDIT = {
        'downstream_min_history_for_window': int(min_hist_for_downstream),
        'synthetic_min_points_per_ticker': int(min_points_per_synth_ticker),
        'synthetic_rows': int(len(synthetic_df)),
        'synthetic_tickers': int(ticker_counts.shape[0]),
        'eligible_synthetic_tickers': int(len(eligible_tickers)),
        'eligible_synthetic_rows': int(eligible_rows),
        'eligible_synthetic_row_ratio': float(eligible_ratio),
        'short_synthetic_entities': int(short_entity_total),
        'synthetic_entities_total': int(synthetic_entity_total),
        'padding_rows_added': int(total_padding_rows_added),
        'min_ticker_length': int(ticker_counts.min()) if len(ticker_counts) > 0 else 0,
        'max_ticker_length': int(ticker_counts.max()) if len(ticker_counts) > 0 else 0,
        'force_all_tickers_eligible': bool(force_all_tickers_eligible)
    }

    print("\nSynthetic downstream eligibility audit:")
    print(TIMEGAN_SYNTH_ENTITY_AUDIT)

TIMEGAN_DATE_BOUND_STATS = {
    'train_date_min': str(train_date_min.date()),
    'train_date_max': str(train_date_max.date()),
    'synthetic_date_clamped_rows': int(date_clamp_total),
}

TIMEGAN_GENERATION_LOG = gen_log
print(f"Generation log entries: {len(TIMEGAN_GENERATION_LOG)}")

## 9. Quality Filtering

In [ ]:
def filter_synthetic_samples(
    synthetic_df,
    train_df,
    numeric_cols,
    quantile_range=(0.01, 0.99)
):
    """Filter synthetic samples with quantile clipping and duplicate removal."""
    if len(synthetic_df) == 0:
        return synthetic_df

    initial_count = len(synthetic_df)
    filtered_df = synthetic_df.copy()

    print(f"\nFiltering with quantile range: {quantile_range}")

    # 1) Clip numeric values to train quantiles
    for col in numeric_cols:
        if col in filtered_df.columns and col in train_df.columns:
            lower = train_df[col].quantile(quantile_range[0])
            upper = train_df[col].quantile(quantile_range[1])
            filtered_df[col] = pd.to_numeric(filtered_df[col], errors='coerce').clip(lower, upper)

    # 2) Drop duplicated synthetic rows (DISABLED to preserve time-series sequence integrity)
    duplicates = 0
    # filtered_df = filtered_df.drop_duplicates().reset_index(drop=True)

    final_count = len(filtered_df)
    removed = initial_count - final_count
    removed_pct = 100.0 * removed / max(initial_count, 1)

    print("\nFiltering summary:")
    print(f"  Initial: {initial_count}")
    print(f"  After filtering: {final_count}")
    print(f"  Removed: {removed} ({removed_pct:.1f}%)")
    print(f"  Duplicate rows removed: {duplicates}")

    return filtered_df

# Apply filtering
if len(synthetic_df) > 0:
    q_low = float(CONFIG.get('quality_quantile_low', 0.01))
    q_high = float(CONFIG.get('quality_quantile_high', 0.99))

    # Ensure generated rows obey train schema constraints before quality filtering
    synthetic_df = postprocess_synthetic_rows(
        synthetic_df,
        numeric_cols=TIMEGAN_NUMERIC_COLUMNS,
        categorical_cols=TIMEGAN_CATEGORICAL_COLUMNS
    )

    synthetic_df_filtered = filter_synthetic_samples(
        synthetic_df=synthetic_df,
        train_df=train,
        numeric_cols=TIMEGAN_NUMERIC_COLUMNS,
        quantile_range=(q_low, q_high)
    )
    print("\nQuality filtering completed")
else:
    synthetic_df_filtered = synthetic_df

In [ ]:
# Additional quality summary for TimeGAN generation
if len(synthetic_df_filtered) > 0:
    print(f"\n{'='*70}")
    print("TIMEGAN GENERATION SUMMARY")
    print(f"{'='*70}")

    synthetic_counts = synthetic_df_filtered[CONFIG['target_column']].value_counts().sort_index()
    real_counts = train[CONFIG['target_column']].value_counts().sort_index()

    summary_df = pd.DataFrame({
        'real_train_count': real_counts,
        'synthetic_count': synthetic_counts
    }).fillna(0).astype(int)
    summary_df['augmented_count'] = summary_df['real_train_count'] + summary_df['synthetic_count']
    summary_df['synthetic_share_in_augmented'] = (
        summary_df['synthetic_count'] / summary_df['augmented_count'].replace(0, np.nan)
    ).fillna(0.0).round(4)

    print(summary_df)

    summary_path = REPORTS_DIR / 'timegan_generation_summary.csv'
    summary_df.to_csv(summary_path, index=True)
    print(f"\n Generation summary saved: {summary_path}")
else:
    print("\nĂ¢ÂÂ  No synthetic samples to summarize")

## 10. Synthetic Data Quality Analysis

In [ ]:
def analyze_synthetic_quality(real_df, synthetic_df, numeric_cols, categorical_cols, target_col):
    """Analyze quality of synthetic data vs real data"""
    if len(synthetic_df) == 0:
        print("No synthetic data to analyze")
        return {}

    results = {}

    print("\n=== Synthetic Data Quality Analysis ===")

    # 1. KS test for numeric columns
    print("\nKolmogorov-Smirnov Test (Numeric Features):")
    print(f"{'Column':<30} {'KS Statistic':>15} {'p-value':>15} {'Similar?':>10}")
    print("-" * 75)

    for col in numeric_cols[:10]:  # Show first 10
        if col in synthetic_df.columns:
            ks_stat, ks_pval = stats.ks_2samp(real_df[col], synthetic_df[col])
            similar = "Yes" if ks_pval > 0.05 else "No"
            results[f'{col}_ks_stat'] = float(ks_stat)
            results[f'{col}_ks_pval'] = float(ks_pval)
            print(f"{col:<30} {ks_stat:>15.4f} {ks_pval:>15.4f} {similar:>10}")

    # 2. JS divergence for categorical columns
    if categorical_cols:
        print("\nJensen-Shannon Divergence (Categorical Features):")
        print(f"{'Column':<30} {'JS Divergence':>15} {'Similar?':>10}")
        print("-" * 60)

        for col in categorical_cols:
            if col in synthetic_df.columns and col != target_col:
                real_dist = real_df[col].value_counts(normalize=True).sort_index()
                syn_dist = synthetic_df[col].value_counts(normalize=True).sort_index()

                # Align distributions
                all_cats = sorted(set(real_dist.index) | set(syn_dist.index))
                real_aligned = np.array([real_dist.get(c, 0) for c in all_cats])
                syn_aligned = np.array([syn_dist.get(c, 0) for c in all_cats])

                js_div = jensenshannon(real_aligned, syn_aligned)
                similar = "Yes" if js_div < 0.1 else "No"
                results[f'{col}_js_div'] = float(js_div)
                print(f"{col:<30} {js_div:>15.4f} {similar:>10}")

    # 3. Target distribution comparison
    print("\n=== Target Distribution Comparison ===")
    real_target = real_df[target_col].value_counts(normalize=True).sort_index()
    syn_target = synthetic_df[target_col].value_counts(normalize=True).sort_index()

    comparison_df = pd.DataFrame({
        'Real': real_target,
        'Synthetic': syn_target
    }).fillna(0)
    print(comparison_df)

    # 4. Temporal consistency check (lag-1 autocorrelation by entity)
    date_col = CONFIG.get('date_column', 'rating_date')
    entity_col = CONFIG.get('entity_column', 'ticker')

    def mean_entity_lag1_autocorr(df_in, value_col):
        if value_col not in df_in.columns:
            return np.nan

        if entity_col not in df_in.columns or date_col not in df_in.columns:
            s = pd.to_numeric(df_in[value_col], errors='coerce').dropna()
            if len(s) < 3 or s.nunique() <= 1:
                return np.nan
            return float(s.autocorr(lag=1))

        tmp = df_in[[entity_col, date_col, value_col]].copy()
        tmp[date_col] = pd.to_datetime(tmp[date_col], errors='coerce')
        vals = []
        for _, grp in tmp.groupby(entity_col):
            g = grp.sort_values(date_col)
            s = pd.to_numeric(g[value_col], errors='coerce').dropna()
            if len(s) >= 3 and s.nunique() > 1:
                ac = s.autocorr(lag=1)
                if pd.notna(ac):
                    vals.append(float(ac))

        if len(vals) == 0:
            return np.nan
        return float(np.mean(vals))

    print("\nTemporal consistency (lag-1 autocorrelation drift):")
    print(f"{'Column':<30} {'Real AC(1)':>12} {'Synth AC(1)':>12} {'Abs Drift':>12}")
    print("-" * 72)

    for col in numeric_cols[:8]:
        if col in real_df.columns and col in synthetic_df.columns:
            ac_real = mean_entity_lag1_autocorr(real_df, col)
            ac_syn = mean_entity_lag1_autocorr(synthetic_df, col)
            if np.isfinite(ac_real) and np.isfinite(ac_syn):
                drift = abs(ac_real - ac_syn)
                results[f'{col}_lag1_autocorr_real'] = float(ac_real)
                results[f'{col}_lag1_autocorr_synth'] = float(ac_syn)
                results[f'{col}_lag1_autocorr_abs_drift'] = float(drift)
                print(f"{col:<30} {ac_real:>12.4f} {ac_syn:>12.4f} {drift:>12.4f}")

    return results

# Analyze quality
if len(synthetic_df_filtered) > 0:
    quality_results = analyze_synthetic_quality(
        train,
        synthetic_df_filtered,
        FEATURE_NUMERIC,
        FEATURE_CATEGORICAL,
        CONFIG['target_column']
    )

    # Save quality results
    with open(MODELS_DIR / 'quality_analysis.json', 'w', encoding='utf-8') as f:
        json.dump(quality_results, f, indent=2)

## 11. Create Augmented Training Set

In [ ]:
# Combine real and synthetic data
if len(synthetic_df_filtered) > 0:
    # Add flag to distinguish real vs synthetic
    train_real = train.copy()
    train_real['is_synthetic'] = 0

    synthetic_final = synthetic_df_filtered.copy()
    synthetic_final['is_synthetic'] = 1

    # Align synthetic schema to real train schema
    for col in train_real.columns:
        if col not in synthetic_final.columns:
            if pd.api.types.is_numeric_dtype(train_real[col]):
                synthetic_final[col] = np.nan
            else:
                synthetic_final[col] = '__SYNTHETIC__'

    # Keep same column order as real data
    synthetic_final = synthetic_final[train_real.columns]

    # Align synthetic dtypes to real train schema for downstream compatibility
    for col in train_real.columns:
        if col == 'is_synthetic':
            continue
        if pd.api.types.is_numeric_dtype(train_real[col]):
            synthetic_final[col] = pd.to_numeric(synthetic_final[col], errors='coerce')
            if pd.api.types.is_integer_dtype(train_real[col]):
                if train_real[col].dropna().empty:
                    fill_value = 0
                else:
                    fill_value = int(train_real[col].mode(dropna=True).iloc[0])
                synthetic_final[col] = synthetic_final[col].round().fillna(fill_value)
                synthetic_final[col] = synthetic_final[col].astype(train_real[col].dtype)
        else:
            synthetic_final[col] = synthetic_final[col].astype(str)

    # Combine
    train_augmented = pd.concat([train_real, synthetic_final], ignore_index=True)

    # Shuffle
    train_augmented = train_augmented.sample(frac=1, random_state=RANDOM_SEED).reset_index(drop=True)

    print("\n=== Augmented Training Set ===")
    print(f"Real samples: {len(train_real)}")
    print(f"Synthetic samples: {len(synthetic_final)}")
    print(f"Total: {len(train_augmented)}")
    print(f"Synthetic ratio: {100*len(synthetic_final)/len(train_augmented):.1f}%")

    # Class distribution after augmentation
    print("\n=== Class Distribution After Augmentation ===")
    print(train_augmented[CONFIG['target_column']].value_counts().sort_index())

else:
    train_augmented = train.copy()
    train_augmented['is_synthetic'] = 0
    print("\nĂ¢ÂÂ  No augmentation applied (no synthetic samples)")

## 11.5 Hybrid Rebalancing (Optional)

**Strategy:** rebalance class distribution while preserving all real observations.
Majority classes are trimmed on synthetic rows first; minority classes can be floor-padded if needed.

In [ ]:
# Hybrid Rebalancing v2 (ENABLED by default)
# Changes vs v1:
# - ENABLE_HYBRID_REBALANCING: True (was False)
# - majority_cap_percentile: 80 (was 75) â€” less aggressive capping
# - minority_floor: 150 (was 200) â€” realistic for rare classes
# - When downsampling majority: prioritize keeping real data over synthetic
# - Never downsample real majority rows; cap synthetic only

ENABLE_HYBRID_REBALANCING = bool(CONFIG.get('enable_hybrid_rebalancing', True))
CAP_PCT = int(CONFIG.get('majority_cap_percentile', 80))
MIN_FLOOR = int(CONFIG.get('minority_floor', 150))

if ENABLE_HYBRID_REBALANCING and len(train_augmented) > 0:
    print("=" * 65)
    print("HYBRID REBALANCING v2  (enabled by default)")
    print("=" * 65)

    tc = CONFIG['target_column']
    pcls = set(int(c) for c in CONFIG.get('priority_class_labels', []))
    aug_c = train_augmented[tc].value_counts().sort_index()

    # Determine the cap from non-priority class distribution only
    np_cnt = aug_c[[i for i in aug_c.index if i not in pcls]]
    nz_np = np_cnt[np_cnt > 0]
    if len(nz_np) > 0:
        cap = int(np.percentile(nz_np.values, CAP_PCT))
    else:
        nz_all = aug_c[aug_c > 0]
        cap = int(nz_all.quantile(0.80)) if len(nz_all) > 0 else MIN_FLOOR * 4
    cap = max(cap, MIN_FLOOR * 2)

    print(f"  Non-priority P{CAP_PCT} cap: {cap}")
    print(f"  Minority floor: {MIN_FLOOR}")
    print(f"  Priority classes (not capped): {sorted(pcls)}")

    chunks = []
    for lbl in sorted(train_augmented[tc].unique()):
        cd = train_augmented[train_augmented[tc] == lbl].copy()
        n = len(cd)
        is_p = int(lbl) in pcls

        if not is_p and n > cap:
            # Downsample majority: keep all real data, trim synthetic first.
            if 'is_synthetic' in cd.columns:
                real_d = cd[cd['is_synthetic'] == 0]
                syn_d = cd[cd['is_synthetic'] == 1]
            else:
                real_d = cd
                syn_d = pd.DataFrame()

            n_real = len(real_d)
            if n_real >= cap:
                cd = real_d.copy()
                print(
                    f"  Cls {lbl} (majority): {n} -> {len(cd)} "
                    "(kept all real rows; trimmed synthetic only)"
                )
            else:
                n_need = cap - n_real
                if n_need > 0 and len(syn_d) > 0:
                    syn_kept = syn_d.sample(
                        n=min(n_need, len(syn_d)), random_state=RANDOM_SEED
                    )
                    cd = pd.concat([real_d, syn_kept], ignore_index=True)
                else:
                    cd = real_d.copy()
                print(
                    f"  Cls {lbl} (majority): {n} -> {len(cd)} "
                    f"(kept all real; synthetic cap add={max(n_need, 0)})"
                )

        elif n < MIN_FLOOR and n > 0:
            # Minority floor via duplication (last resort)
            extra = cd.sample(
                n=MIN_FLOOR - n, random_state=RANDOM_SEED, replace=True
            )
            if 'is_synthetic' in extra.columns:
                extra['is_synthetic'] = 1
            cd = pd.concat([cd, extra], ignore_index=True)
            print(f"  Cls {lbl} (minority): {n} -> {len(cd)} (floor padded)")

        else:
            status = "priority-kept" if is_p else "kept"
            print(f"  Cls {lbl}: {n} ({status})")

        chunks.append(cd)

    train_augmented = pd.concat(chunks, ignore_index=True)
    train_augmented = train_augmented.sample(
        frac=1, random_state=RANDOM_SEED
    ).reset_index(drop=True)

    fc = train_augmented[tc].value_counts().sort_index()
    nzf = fc[fc > 0]
    rat = nzf.min() / nzf.max() if len(nzf) > 0 and nzf.max() > 0 else float('nan')

    print(f"\n  After rebalancing: {len(train_augmented)} samples")
    print(f"  Class range: [{nzf.min()}, {nzf.max()}]")
    if np.isfinite(rat):
        ql = ('EXCELLENT' if rat > 0.5 else
              'GOOD' if rat > 0.3 else
              'MODERATE' if rat > 0.1 else 'STILL IMBALANCED')
        print(f"  Imbalance ratio: {rat:.3f}  ({ql})")
    else:
        print("  Imbalance ratio: N/A")

    print(f"\n  Full distribution after rebalancing:")
    print(fc)

else:
    print("\nHybrid rebalancing: SKIPPED (disabled in CONFIG or no data)")

In [ ]:
# Visualize before/after augmentation
full_class_index = list(range(int(CONFIG['target_min_label']), int(CONFIG['target_max_label']) + 1))

before_counts = (
    train[CONFIG['target_column']]
    .value_counts()
    .sort_index()
    .reindex(full_class_index, fill_value=0)
    .astype(int)
)
after_counts = (
    train_augmented[CONFIG['target_column']]
    .value_counts()
    .sort_index()
    .reindex(full_class_index, fill_value=0)
    .astype(int)
)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Before
before_counts.plot(kind='bar', ax=axes[0], color='steelblue')
axes[0].set_title('Before TimeGAN Augmentation')
axes[0].set_xlabel('Class')
axes[0].set_ylabel('Count')

# After
after_counts.plot(kind='bar', ax=axes[1], color='coral')
axes[1].set_title('After TimeGAN Augmentation')
axes[1].set_xlabel('Class')
axes[1].set_ylabel('Count')

plt.tight_layout()
plt.savefig(REPORTS_DIR / 'class_distribution_timegan_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 12. Save Augmented Training Set

In [ ]:
# Save augmented training set
output_file_timegan = SPLITS_DIR / 'train_augmented_timegan.csv'
output_file_compat = SPLITS_DIR / 'train_augmented_ctgan.csv'  # compatibility alias

# Fail-fast temporal assertion: synthetic rating_date must stay inside active train-fold range
dtc = CONFIG.get('date_column', 'rating_date')
ent = CONFIG.get('entity_column', 'ticker')
min_hist_for_downstream = int(CONFIG.get('downstream_min_history_for_window', 9))
min_eligible_ratio = float(CONFIG.get('downstream_audit_min_synth_window_ratio', 0.10))
enforce_all_eligible = bool(CONFIG.get('downstream_enforce_all_synthetic_tickers_eligible', True))

TIMEGAN_SYNTH_ENTITY_AUDIT_FINAL = {}
if dtc in train_augmented.columns:
    train_augmented[dtc] = pd.to_datetime(train_augmented[dtc], errors='coerce')

if 'is_synthetic' in train_augmented.columns and dtc in train_augmented.columns and dtc in train.columns:
    syn_mask = train_augmented['is_synthetic'] == 1
    if int(syn_mask.sum()) > 0:
        train_dates = pd.to_datetime(train[dtc], errors='coerce').dropna()
        syn_dates = pd.to_datetime(train_augmented.loc[syn_mask, dtc], errors='coerce')

        if len(train_dates) == 0:
            raise ValueError(f"Train {dtc} is empty after datetime parsing; cannot validate synthetic dates")

        train_min = train_dates.min()
        train_max = train_dates.max()
        invalid_mask = syn_dates.isna() | (syn_dates < train_min) | (syn_dates > train_max)
        invalid_count = int(invalid_mask.sum())

        if invalid_count > 0:
            bad_examples = syn_dates[invalid_mask].astype(str).head(5).tolist()
            raise ValueError(
                f"Detected {invalid_count} synthetic rows with invalid {dtc} outside "
                f"[{train_min.date()} -> {train_max.date()}]. Examples: {bad_examples}"
            )

        print(
            f"Temporal assertion passed: all synthetic {dtc} are within "
            f"[{train_min.date()} -> {train_max.date()}]"
        )

        # Downstream-window eligibility assertion
        if ent in train_augmented.columns:
            synth_df = train_augmented.loc[syn_mask].copy()
            synth_df[dtc] = pd.to_datetime(synth_df[dtc], errors='coerce')

            ticker_counts = synth_df[ent].astype(str).value_counts()
            eligible_tickers = ticker_counts[ticker_counts >= min_hist_for_downstream].index.tolist()
            ineligible_tickers = ticker_counts[ticker_counts < min_hist_for_downstream].index.tolist()
            eligible_rows = int(synth_df[ent].astype(str).isin(eligible_tickers).sum())
            eligible_ratio = float(eligible_rows / max(len(synth_df), 1))

            # Monotonic date continuity check for each synthetic ticker
            non_monotonic_tickers = []
            for ticker_value, g in synth_df.groupby(ent):
                gd = g[dtc].dropna().sort_values()
                if len(gd) <= 1:
                    continue
                diffs = gd.diff().dt.days.dropna()
                if len(diffs) > 0 and int((diffs <= 0).sum()) > 0:
                    non_monotonic_tickers.append(str(ticker_value))

            TIMEGAN_SYNTH_ENTITY_AUDIT_FINAL = {
                'downstream_min_history_for_window': int(min_hist_for_downstream),
                'min_eligible_ratio_required': float(min_eligible_ratio),
                'enforce_all_eligible': bool(enforce_all_eligible),
                'synthetic_rows': int(len(synth_df)),
                'synthetic_tickers': int(ticker_counts.shape[0]),
                'eligible_synthetic_tickers': int(len(eligible_tickers)),
                'ineligible_synthetic_tickers': int(len(ineligible_tickers)),
                'eligible_synthetic_rows': int(eligible_rows),
                'eligible_synthetic_row_ratio': float(eligible_ratio),
                'min_ticker_length': int(ticker_counts.min()) if len(ticker_counts) > 0 else 0,
                'max_ticker_length': int(ticker_counts.max()) if len(ticker_counts) > 0 else 0,
                'non_monotonic_ticker_count': int(len(non_monotonic_tickers))
            }

            print("Synthetic eligibility assertion summary:")
            print(TIMEGAN_SYNTH_ENTITY_AUDIT_FINAL)

            if len(non_monotonic_tickers) > 0:
                raise ValueError(
                    "Synthetic ticker date continuity check failed: found non-monotonic timestamps. "
                    f"Examples: {non_monotonic_tickers[:5]}"
                )

            if enforce_all_eligible:
                if len(ineligible_tickers) > 0:
                    raise ValueError(
                        "Detected synthetic tickers shorter than downstream minimum history. "
                        f"ineligible={len(ineligible_tickers)} with threshold={min_hist_for_downstream}. "
                        "Increase synthetic_min_points_per_ticker or reduce ticker fragmentation."
                    )
                print(
                    "Strict synthetic eligibility passed: all synthetic tickers "
                    f"have >= {min_hist_for_downstream} records"
                )
            else:
                if eligible_ratio < min_eligible_ratio:
                    raise ValueError(
                        "Synthetic ticker histories are too short for downstream windowing. "
                        f"eligible_ratio={eligible_ratio:.4f} < required={min_eligible_ratio:.4f}. "
                        "Increase synthetic_min_points_per_ticker, seq_len, or reduce row fragmentation."
                    )

                print(
                    "Synthetic eligibility assertion passed: "
                    f"eligible_ratio={eligible_ratio:.4f} >= required={min_eligible_ratio:.4f}"
                )

if dtc in train_augmented.columns:
    train_augmented[dtc] = train_augmented[dtc].dt.strftime('%Y-%m-%d')

train_augmented.to_csv(output_file_timegan, index=False)
train_augmented.to_csv(output_file_compat, index=False)

print(f"\nAugmented training set saved to: {output_file_timegan}")
print(f"Compatibility copy saved to: {output_file_compat}")
print(f"  Total records: {len(train_augmented)}")
print(f"  Real: {(train_augmented['is_synthetic']==0).sum()}")
print(f"  Synthetic: {(train_augmented['is_synthetic']==1).sum()}")

## 13. Generate Report

In [ ]:
# Generate comprehensive report
if 'TIMEGAN_GENERATION_LOG' not in globals():
    print(" TIMEGAN_GENERATION_LOG not found. Generation cell may not have run.")
generation_log = TIMEGAN_GENERATION_LOG if 'TIMEGAN_GENERATION_LOG' in globals() else []
trained_class_models = [
    cls for cls, info in TIMEGAN_TRAINING_REGISTRY.get('per_class', {}).items()
    if info.get('status') == 'trained'
]

global_registry = TIMEGAN_TRAINING_REGISTRY.get('global', {})
global_windows_used = global_registry.get('n', global_registry.get('windows_used', 'N/A'))
split_info = TRAIN_VAL_TEST_SPLIT_INFO if 'TRAIN_VAL_TEST_SPLIT_INFO' in globals() else {}

split_method = split_info.get('split_method', 'unknown')
if split_method == 'walk_forward_expanding':
    split_description = (
        f"Walk-forward expanding window | "
        f"active_fold={split_info.get('active_fold_id', 'N/A')} / {split_info.get('n_folds', 'N/A')} | "
        f"train_years={split_info.get('train_years', [])} | "
        f"val_years={split_info.get('val_years', [])} | "
        f"test_years={split_info.get('test_years', [])}"
    )
else:
    split_description = (
        f"Random split | "
        f"train={split_info.get('train_ratio', CONFIG.get('train_ratio', 'N/A'))} | "
        f"val={split_info.get('val_ratio', CONFIG.get('val_ratio', 'N/A'))} | "
        f"test={split_info.get('test_ratio', CONFIG.get('test_ratio', 'N/A'))}"
    )

if 'is_synthetic' in train_augmented.columns:
    final_synth_count = int((train_augmented['is_synthetic'] == 1).sum())
else:
    final_synth_count = int(max(len(train_augmented) - len(train), 0))
final_synth_ratio = (100.0 * final_synth_count / len(train_augmented)) if len(train_augmented) > 0 else 0.0

report = f"""# TimeGAN Augmentation Report (Standalone Pipeline)

**Generated:** {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}

## Configuration
- **Random seed:** {CONFIG['random_seed']}
- **Target column:** {CONFIG['target_column']}
- **Split strategy:** {split_description}
- **Sequence length (`seq_len`):** {CONFIG['seq_len']}
- **Sequence stride:** {CONFIG['sequence_stride']}
- **TimeGAN train steps:** {CONFIG['timegan_train_steps']}
- **Batch size:** {CONFIG['timegan_batch_size']}
- **Learning rate:** {CONFIG['timegan_learning_rate']}
- **Use GPU (requested):** {CONFIG.get('timegan_use_gpu', False)}

## Pipeline Mode
 Standalone mode: loaded source dataset and split inside notebook
 Sequence-aware mode: TimeGAN windows grouped by entity and ordered by time

## Data Summary
- **Original train size:** {len(train)}
- **Val size:** {len(val)}
- **Test size:** {len(test)}
- **Synthetic rows generated (raw):** {len(synthetic_df) if len(synthetic_df) > 0 else 0}
- **Synthetic rows after filtering:** {len(synthetic_df_filtered) if len(synthetic_df_filtered) > 0 else 0}
- **Final augmented train size:** {len(train_augmented)}
- **Synthetic rows in final augmented set:** {final_synth_count}
- **Synthetic ratio:** {final_synth_ratio:.1f}%

## TimeGAN Training Summary
- **Global model status:** {global_registry.get('status', 'unknown')}
- **Global windows used:** {global_windows_used}
- **Per-class trained models:** {trained_class_models if len(trained_class_models) > 0 else 'None'}

## Generation Log
"""

if len(generation_log) == 0:
    report += "\n- No generation log entries available."
else:
    for item in generation_log:
        report += (
            f"\n- Class {item.get('class_label')}: "
            f"requested {item.get('windows_requested')} windows, "
            f"generated {item.get('windows_generated')} windows, "
            f"model source = {item.get('model_source')}, "
            f"JS rejected = {item.get('js_rejected', 0)}, "
            f"RF rejected = {item.get('rf_rejected', 0)}"
        )

report += """

## Class Distribution

### Before Augmentation
"""

before_dist = train[CONFIG['target_column']].value_counts().sort_index()
for class_label, count in before_dist.items():
    report += f"\n- Class {class_label}: {count}"

report += "\n\n### After Augmentation\n"

after_dist = train_augmented[CONFIG['target_column']].value_counts().sort_index()
for class_label, count in after_dist.items():
    original = before_dist.get(class_label, 0)
    increase = count - original
    pct = 100 * increase / original if original > 0 else 0
    report += f"\n- Class {class_label}: {count} (+{increase}, {pct:.1f}%)"

report += f"""

## Preprocessing Applied
 Full preprocessing pipeline executed (imputation, log transform, scaling, encoding) with fit-on-train only.
 TimeGAN-specific min-max scaling applied on model feature columns before sequence training.
 Synthetic sequences inverse-scaled and mapped back to tabular rows.

## Synthetic Data Quality
- Quantile-based clipping applied at ({CONFIG.get('quality_quantile_low', 0.01)}, {CONFIG.get('quality_quantile_high', 0.99)})
- Duplicate synthetic rows are not removed (disabled to preserve sequence integrity)
- See `quality_analysis.json` for statistical comparison between real and synthetic sets

## Anti-Leakage Protocol
 All transformations fitted only on train set
 TimeGAN trained only on train-derived windows
 Synthetic samples added only to train set
 Val/test sets remain real data for evaluation

## Output Files
- `splits/train_raw.csv` - Raw training split
- `splits/val_raw.csv` - Raw validation split
- `splits/test_raw.csv` - Raw test split
- `splits/train.csv` - Training set (preprocessed)
- `splits/val.csv` - Validation set (preprocessed)
- `splits/test.csv` - Test set (preprocessed)
- `splits/train_augmented_timegan.csv` - Augmented training set (primary)
- `splits/train_augmented_ctgan.csv` - Compatibility alias
- `models/timegan/timegan_training_registry.json` - TimeGAN model/training metadata
- `models/timegan/quality_analysis.json` - Quality metrics
- `config_timegan.json` - Configuration file
- `reports/class_distribution_timegan_comparison.png` - Visualization
- `reports/timegan_generation_summary.csv` - Per-class generation summary
- `reports/walk_forward_folds_summary.csv` - Walk-forward fold summary
- `reports/split_metadata.json` - Split metadata

## Next Steps
1. Train downstream classification model on augmented training set
2. Evaluate on unchanged val/test sets
3. Compare baseline vs TimeGAN augmentation metrics (balanced accuracy, macro-F1, ordinal metrics)
4. Analyze minority-class gains and calibration effects

## Reproducibility
 Fixed random seed ({CONFIG['random_seed']})
 Config and generation metadata saved
 Standalone split and processing protocol kept deterministic where possible
"""

# Save report
report_path = REPORTS_DIR / 'timegan_augmentation_report.md'
with open(report_path, 'w', encoding='utf-8') as f:
    f.write(report)

print(" Report generated")
print(f" Report saved to: {report_path}")
print(report)

## 14. Summary

**Completed:**
- â€œ Loaded source data and performed standalone train/val/test split
- â€œ Applied preprocessing pipeline (fit on train only)
- â€œ Built entity-time sequence windows for TimeGAN
- â€œ Trained TimeGAN (global + eligible per-class models)
- â€œ Generated class-focused synthetic sequences
- â€œ Unrolled synthetic sequences back to tabular rows
- â€œ Applied quality filtering
- â€œ Created augmented training set
- â€œ Maintained anti-leakage protocol

**Key Outputs:**
- `splits/train_raw.csv`, `val_raw.csv`, `test_raw.csv` - Raw splits
- `splits/train.csv`, `val.csv`, `test.csv` - Preprocessed splits
- `splits/train_augmented_timegan.csv` - Primary augmented training set
- `splits/train_augmented_ctgan.csv` - Compatibility alias
- `models/timegan/timegan_training_registry.json` - TimeGAN training metadata
- `reports/timegan_augmentation_report.md` - Final report

**Next Steps:**
- Train classification models on both baseline and TimeGAN-augmented data
- Compare TimeGAN vs TabDDPM vs SMOTE vs no augmentation
- Evaluate on the same untouched test split for fair comparison